# Invoice Extraction Pipeline v8.1 — TATR Full-Row Scan Fix

## Root cause of 0 line items (found after v8 run)

The grid WAS being built correctly — `image_to_data()` + word assignment worked.
The problem was that alias matching and positional fallback **only scanned the
first 5–7 rows** of the grid, but TATR treats the entire page as one table, so a
79-row grid has the product table header at row ~35 and data at rows ~36–38.

| What was broken | Fix |
|---|---|
| `_find_best_header_row` scanned rows 0–5 only | Replaced with `_find_best_table_region()` that scans ALL rows |
| `_positional_column_mapping` sampled `grid[1:8]` (7 rows) | Now accepts `row_start`/`row_end` params; called on the numeric-dense window |
| Numeric density threshold 40% | Lowered to 25% (sparse invoice tables have many empty cells) |
| No diagnostic output for grid content | First 5 non-empty rows now printed so you can see what the grid contains |

**Required:** GPU T4×2 · Secret: `HF_TOKEN`

## 1  Install dependencies

In [1]:
!pip install transformers accelerate bitsandbytes --quiet
!pip install pymupdf opencv-python-headless deskew --quiet
!pip install pillow --quiet
!pip install pdfminer.six pdfplumber --quiet
!pip install pytesseract python-dateutil --quiet
!apt-get install -y tesseract-ocr tesseract-ocr-fra --quiet
!pip install camelot-py tabula-py --quiet
!apt-get install -y ghostscript --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 72.8 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.8 MB/s eta 0:00:00:00:01
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-fra
0 upgraded, 1 newly installed, 0 to remove and 134 not upgraded.
Need to get 527 kB of archives.
After this operation, 1,145 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ub

## 2  Imports

In [2]:
import os, io, json, re, tempfile, math
import pandas as pd
import numpy as np
import cv2
import pymupdf
from deskew import determine_skew
from PIL import Image
import pdfplumber
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBox
import pytesseract
import torch

# Compatibility shim for environments where torchvision expects PIL._typing._Ink
try:
    from PIL import _typing as _pil_typing
    if not hasattr(_pil_typing, '_Ink'):
        from typing import Any
        _pil_typing._Ink = Any
except Exception:
    pass

try:
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, pipeline,
        TableTransformerForObjectDetection, DetrImageProcessor,
        BitsAndBytesConfig,
    )
except ImportError as e:
    if '_Ink' in str(e):
        from PIL import _typing as _pil_typing
        from typing import Any
        _pil_typing._Ink = Any
        from transformers import (
            AutoTokenizer, AutoModelForCausalLM, pipeline,
            TableTransformerForObjectDetection, DetrImageProcessor,
            BitsAndBytesConfig,
        )
    else:
        raise

print(f'PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory//1024**3} GB)')

try:
    import camelot
except Exception:
    camelot = None

try:
    import tabula
except Exception:
    tabula = None


try:
    from docling.document_converter import DocumentConverter
except Exception:
    DocumentConverter = None


PyTorch: 2.9.0+cu126  CUDA: True
  GPU 0: Tesla T4 (14 GB)
  GPU 1: Tesla T4 (14 GB)


## 3  Configuration

In [3]:
INVOICE_PATHS = [
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/Direct - AVIKO - G24000153.pdf",
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/Direct - NEMCO - G24000046.pdf",
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/Direct - QUIRCH - G2400003.pdf",
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/G23000461 BARON PHILIPE 2304001154.pdf",
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/G23000461 BARON PHILIPPE 2002075457.pdf",
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/G23000461 DE LADOUCETTE 0033043657.pdf",
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/G23000461 GEORGES DUBOEUF 2023-5275.pdf",
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/G23000461 HOBNOB 2023-5274.pdf",
    "/kaggle/input/datasets/smujtabahussain/invoice-samples/G23000461 MINUTY 00568057.pdf",
]

MODEL_ID           = "meta-llama/Llama-3.1-8B-Instruct"
MAX_NEW_TOKENS     = 1536
MAX_MARKDOWN_CHARS = 14_000
NATIVE_TEXT_CHAR_THRESHOLD = 80
TATR_CONFIDENCE    = 0.55   # lowered further to catch more cells
RASTER_DPI         = 300
TABLE_RASTER_DPI   = 450
SKEW_THRESHOLD     = 0.5

# FIX: 'pu' removed — matched substring of 'Purchase' causing false col mapping.
# FIX: Added Dutch aliases (AVIKO uses Dutch column headers).
COLUMN_ALIASES = {
    "description":     ["description", "designation", "article", "product",
                        "produit", "item", "goods", "libelle", "merchandise",
                        "details", "label",
                        "omschrijving", "artikelomschrijving"],          # Dutch
    "quantity":        ["qty", "quantity", "quantite", "qte", "units",
                        "bottles", "cases", "pcs", "nb", "nbre", "colis",
                        "aantal", "stuks", "colli"],                     # Dutch
    "unit_price":      ["unit price", "prix unit", "prix unitaire", "p.u.",
                        "rate", "unitary", "unit cost", "prix/u",
                        "eenheidsprijs", "e-prijs"],                     # Dutch
    "line_amount":     ["amount", "line total", "ext price", "extended",
                        "montant", "line amount", "total price", "net amount",
                        "total ht", "montant ht",
                        "bedrag", "totaal", "netto bedrag"],             # Dutch
    "ean_code":        ["ean", "barcode", "upc", "gtin",
                        "ean-code", "ean code"],
    "material_number": ["material", "ref", "reference", "sku", "code article",
                        "mat.", "article no", "item no", "part no", "code",
                        "art.nr", "artnr", "artno"],                     # Dutch
}

# Buyer identity strings — anchor containing any of these is discarded
_BUYER_STRINGS = ['caribbean', 'cpj', 'guinep', 'montego', 'buyer']

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Configuration ready.")


Configuration ready.


## 4  HF Token

In [4]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("HF token loaded.")


HF token loaded.


## 5  Ground truth

In [5]:
GROUND_TRUTH = [
    {
        "vendor_name": "Aviko B.V.",
        "vendor_address": "P.O. Box 8 7220 AA Steenderen Holland",
        "invoice_number": "93184726",
        "purchase_order_number": "POH112330",
        "invoice_date": "2024-03-06",
        "sub_total": 33724.44,
        "tax": 0.0,
        "total": 37125.98,
        "shipping_handling_charge": 3033.96,
        "insurance_charge": 367.58,
        "discount": 0.0,
        "ship_to": "Caribbean Producers Jamaica Ltd One Guinep Way Montego Freeport - MONTEGO BAY JAMAICA",
        "bill_to": "Caribbean Producers Jamaica Ltd One Guinep Way Montego Freeport MONTEGO BAY JAMAICA",
        "delivery_note_number": "83661626",
        "iban": None,
        "vendor_phone": "+31 (0)575-458200",
        "line_items": [
            {"ean_code": "8710449910533", "material_number": "802142",
             "description": "CAR Aviko Str. cut fries 3/8\" 4x2500g reg3",
             "quantity": 1380.0, "unit_price": 12.07, "line_amount": 16656.60,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": "8710449939725", "material_number": "806959",
             "description": "CAR Aviko Sweet Potato Fries 3/8 5x5lbs reg3",
             "quantity": 192.0, "unit_price": 40.16, "line_amount": 7710.72,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": "8710449918225", "material_number": "804104",
             "description": "CAR Aviko Hashbrowns Triangles 4x2500g reg3",
             "quantity": 648.0, "unit_price": 14.44, "line_amount": 9357.12,
             "po_number": None, "delivery_note_number": None},
        ],
    },
    {
        "vendor_name": "Nemco Food Trading Inc",
        "vendor_address": "207 Bedford St Lakeville, MA 02347",
        "invoice_number": "93828",
        "purchase_order_number": "POJ000626",
        "invoice_date": "2024-02-06",
        "sub_total": 34011.00,
        "tax": 0.0,
        "total": 34011.00,
        "shipping_handling_charge": 0.0,
        "insurance_charge": 0.0,
        "discount": 0.0,
        "ship_to": "Caribbean Producers JA. Ltd. 1 Guinep Way Montego Freeport Montego Bay Jamaica Amy Davis",
        "bill_to": "Caribbean Producers JA. LTD 1 Guinep Way Montego Freeport Montego Bay Jamaica",
        "delivery_note_number": None,
        "iban": None,
        "vendor_phone": None,
        "line_items": [
            {"ean_code": None, "material_number": "25368A",
             "description": "24/15 CPJ S23 COCONUT MILK EZ OPEN LID",
             "quantity": 1900.0, "unit_price": 14.49, "line_amount": 27531.00,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "25368B",
             "description": "24/15 CPJ COCONUT CREAM EZ OPEN LID",
             "quantity": 360.0, "unit_price": 18.00, "line_amount": 6480.00,
             "po_number": None, "delivery_note_number": None},
        ],
    },
    {
        "vendor_name": "Quirch Foods LLC",
        "vendor_address": "2701 S Le Jeune Rd, 12th Floor Coral Gables FL 33134",
        "invoice_number": "160196",
        "purchase_order_number": "POH111849",
        "invoice_date": "2024-02-07",
        "sub_total": 90807.50,
        "tax": 0.0,
        "total": 98375.00,
        "shipping_handling_charge": 7436.25,
        "insurance_charge": 131.25,
        "discount": 0.0,
        "ship_to": "Caribbean Producers Jamaica, LTD 1 Guinep Way Montego Bay, Jamaica",
        "bill_to": "Caribbean Producers Jamaica, LTD 1 Guinep Way Montego Bay, Jamaica",
        "delivery_note_number": None,
        "iban": None,
        "vendor_phone": "(305) 691-3535",
        "line_items": [
            {"ean_code": None, "material_number": "111111147",
             "description": "CPJ-Shrimp 21/25 Ez Peel Iqf. Marinated. Pack: 10 x 1 Lb",
             "quantity": 7000.0, "unit_price": 2.2982, "line_amount": 16087.40,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "111111149",
             "description": "CPJ-Shrimp 31/40 Ez Peel Iqf. Marinated. Pack: 10 x 1 Lb",
             "quantity": 5000.0, "unit_price": 2.1982, "line_amount": 10991.00,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "111111151",
             "description": "CPJ-Shrimp 41/50 Ez Peel Iqf. Marinated. Pack: 10 x 1 Lb",
             "quantity": 1000.0, "unit_price": 1.7982, "line_amount": 1798.20,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "111111153",
             "description": "CPJ-Shrimp 21/25 Wht Pdto Iqf. Marinated. Pack: 10 x 1 Lb",
             "quantity": 12500.0, "unit_price": 2.5982, "line_amount": 32477.50,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "111111161",
             "description": "CPJ-Shrimp 26/30 Wht Pdto Iqf. Marinated. Pack: 10 x 1 Lb",
             "quantity": 7500.0, "unit_price": 2.3982, "line_amount": 17986.50,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "111143376",
             "description": "CPJ-Shrimp 16/20 Ez Peel Iqf. Marinated. Pack: 10 x 1 Lb",
             "quantity": 4500.0, "unit_price": 2.5482, "line_amount": 11466.90,
             "po_number": None, "delivery_note_number": None},
        ],
    },
    {
        "vendor_name": "BARON PHILIPPE DE ROTHSCHILD, S.A.",
        "vendor_address": "33250 PAUILLAC rue de Grassi - France",
        "invoice_number": "2304001154",
        "purchase_order_number": "H110047",
        "invoice_date": "2023-07-25",
        "sub_total": 1181.32,
        "tax": 0.0,
        "total": 1181.32,
        "shipping_handling_charge": 0.0,
        "insurance_charge": 0.0,
        "discount": 0.0,
        "ship_to": "CARIBBEAN PRODUCERS JAMAICALIMITED 1 GUINEP WAY, FEEPORT 00000 MONTEGO BAY Jamaïque",
        "bill_to": "CARIBBEAN PRODUCERS JAMAICALIMITED 1 GUINEP WAY, FEEPORT 00000 MONTEGO BAY Jamaïque",
        "delivery_note_number": None,
        "iban": None,
        "vendor_phone": "+33 (0)5 56 73 20 20",
        "line_items": [
            {"ean_code": None, "material_number": "28",
             "description": "MOUTON CADET ROUGE 2020",
             "quantity": 28.0, "unit_price": 42.19, "line_amount": 1181.32,
             "po_number": None, "delivery_note_number": None},
        ],
    },
    {
        "vendor_name": "BARON PHILIPPE DE ROTHSCHILD MAIΡΟ CHILE S.A",
        "vendor_address": "FUNDO VINA MAIPO LOTE A HIJUELAS NORTE 00000 BUIN SANTIAGO DE CHILE Chili",
        "invoice_number": "2002075457",
        "purchase_order_number": "POH110216",
        "invoice_date": "2023-07-13",
        "sub_total": 168.31,
        "tax": 0.0,
        "total": 168.31,
        "shipping_handling_charge": 0.0,
        "insurance_charge": 0.0,
        "discount": 0.0,
        "ship_to": "CARIBBEAN PRODUCERS JAMAICALIMITED 1 GUINEP WAY, FEEPORT 00000 MONTEGO BAY Jamaïque",
        "bill_to": "BARON PHILIPPE DE ROTHSCHILD MAIΡΟ CHILE S.A FUNDO VINA MAIPO LOTE A HIJUELAS NORTE 00000 BUIN SANTIAGO DE CHILE Chili",
        "delivery_note_number": None,
        "iban": None,
        "vendor_phone": None,
        "line_items": [
            {"ean_code": None, "material_number": "1",
             "description": "DUMMY Format 3L ESCUDO ROJO GRAN RESERVA",
             "quantity": 1.0, "unit_price": 12.41, "line_amount": 12.41,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "20",
             "description": "ESCUDO ROJO SOMMELIER'S CORKSCREW",
             "quantity": 20.0, "unit_price": 1.89, "line_amount": 37.80,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "30",
             "description": "ESCUDO ROJO 1 BOTTLE BAG",
             "quantity": 30.0, "unit_price": 1.77, "line_amount": 53.10,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "1",
             "description": "ESCUDO ROJO ROLL UP",
             "quantity": 1.0, "unit_price": 65.00, "line_amount": 65.00,
             "po_number": None, "delivery_note_number": None},
        ],
    },
    {
        "vendor_name": "de Ladoucette",
        "vendor_address": "Château du Nozet 58150 POUILLY SUR LOIRE",
        "invoice_number": "0033043657",
        "purchase_order_number": "PO H110077",
        "invoice_date": "2023-07-21",
        "sub_total": 2310.00,
        "tax": 0.0,
        "total": 2310.00,
        "shipping_handling_charge": 0.0,
        "insurance_charge": 0.0,
        "discount": 0.0,
        "ship_to": "CARIBBEAN PRODUCERS 1 GUINEP WAY UNIT #2 LOT FREEPORT CENTER MONTEGO BAY JAMAIQUE",
        "bill_to": "CARIBBEAN PRODUCERS 1 GUINEP WAY UNIT #2 LOT FREEPORT CENTER MONTEGO BAY JAMAIQUE",
        "delivery_note_number": None,
        "iban": "FR76 10096185500001814170155",
        "vendor_phone": "03 86 39 10 16",
        "line_items": [
            {"ean_code": None, "material_number": "14",
             "description": "DE LADOUCETTE POUILLY FUME 2022",
             "quantity": 84.0, "unit_price": 14.40, "line_amount": 1209.60,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "14",
             "description": "SANCERRE COMTE LAFOND ROUGE 2021",
             "quantity": 84.0, "unit_price": 13.10, "line_amount": 1100.40,
             "po_number": None, "delivery_note_number": None},
        ],
    },
    {
        "vendor_name": "LES VINS GEORGES DUBOEUF",
        "vendor_address": "208 Rue de Lancié 71570 ROMANECHE-THORINS - FRANCE",
        "invoice_number": "2023/5275",
        "purchase_order_number": "PC#H110051",
        "invoice_date": "2023-07-26",
        "sub_total": 1464.96,
        "tax": 0.0,
        "total": 1464.96,
        "shipping_handling_charge": 0.0,
        "insurance_charge": 0.0,
        "discount": 0.0,
        "ship_to": "CARIBBEAN PRODUCERS JA.LTD 1 GUINEP WAY FREEPORT MONTEGO BAY JAMAICA WI",
        "bill_to": "CARIBBEAN PRODUCERS JA.LTD 1 GUINEP WAY FREEPORT MONTEGO BAY JAMAICA WI",
        "delivery_note_number": None,
        "iban": "FR76 30003 01210 00020118018 60",
        "vendor_phone": None,
        "line_items": [
            {"ean_code": None, "material_number": "C837",
             "description": "VIN DE FRANCE ROUGE 13,0% PINOT NOIR - ECUSSON 2022",
             "quantity": 336.0, "unit_price": 4.36, "line_amount": 1464.96,
             "po_number": None, "delivery_note_number": None},
        ],
    },
    {
        "vendor_name": "HOBNOB CELLARS",
        "vendor_address": "208 Rue de Lancié 71570 ROMANECHE-THORINS FRANCE",
        "invoice_number": "2023/5274",
        "purchase_order_number": "PO H110052",
        "invoice_date": "2023-07-26",
        "sub_total": 3302.88,
        "tax": 0.0,
        "total": 3302.88,
        "shipping_handling_charge": 0.0,
        "insurance_charge": 0.0,
        "discount": 0.0,
        "ship_to": "CARIBBEAN PRODUCERS JA.LTD 1 GUINEP WAY FREEPORT MONTEGO BAY JAMAICA WI",
        "bill_to": "CARIBBEAN PRODUCERS JA.LTD 1 GUINEP WAY FREEPORT MONTEGO BAY JAMAICA WI",
        "delivery_note_number": None,
        "iban": "FR76 30003 01210 00020118018 60",
        "vendor_phone": None,
        "line_items": [
            {"ean_code": None, "material_number": "K975",
             "description": "PINOT NOIR PAYS D'OC IGP HOBNOB - ST 13,0% 2022",
             "quantity": 336.0, "unit_price": 5.10, "line_amount": 1713.60,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "K975",
             "description": "P NOT NOIR PAYS D'OC IGP HOBNOB - ST 2022 13,0%",
             "quantity": 48.0, "unit_price": 0.0, "line_amount": 0.0,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "g995",
             "description": "CHARDONNAY PAYS D'OC 13,0% IGP HOBNOB - ST 2022",
             "quantity": 336.0, "unit_price": 4.73, "line_amount": 1589.28,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "g995",
             "description": "CHARDONNAY PAYS D'OC 13,0% IGPHOBNOB - ST 2022",
             "quantity": 48.0, "unit_price": 0.0, "line_amount": 0.0,
             "po_number": None, "delivery_note_number": None},
        ],
    },
    {
        "vendor_name": "MINUTY S.A.S.",
        "vendor_address": "2491 route de la Berle 83580 Gassin - France",
        "invoice_number": "00568057",
        "purchase_order_number": "PO H110049",
        "invoice_date": "2023-07-08",
        "sub_total": 12960.00,
        "tax": 0.0,
        "total": 12960.00,
        "shipping_handling_charge": 0.0,
        "insurance_charge": 0.0,
        "discount": 0.0,
        "ship_to": "CARIBBEAN PRODUCERS 1 GUINEP WAY FREEPORT MONTEGO BAY JAMAICA WI",
        "bill_to": "CARIBBEAN PRODUCERS 1 GUINEP WAY FREEPORT MONTEGO BAY JAMAICA WI",
        "delivery_note_number": None,
        "iban": "FR76 1009 6185 7400 0564 4680 104",
        "vendor_phone": "04 94 56 12 09",
        "line_items": [
            {"ean_code": None, "material_number": "022E 06",
             "description": "CHATEAU MINUTY OR A.O.P COTES DE PROVENCE ROSE ET OR ROSE 0,75 L, ALCOHOL 12,5% 2022 BOUTEILLES CRD EXPORT",
             "quantity": 180.0, "unit_price": 11.00, "line_amount": 1980.00,
             "po_number": None, "delivery_note_number": None},
            {"ean_code": None, "material_number": "222E 12",
             "description": "M DE MINUTY AOP COTES DE PROVENCE M DE MINUTY ROSE 0,75 L, ALCOHOL 13% 2022 BOUTEILLES CRD EXPORT",
             "quantity": 1800.0, "unit_price": 6.10, "line_amount": 10980.00,
             "po_number": None, "delivery_note_number": None},
        ],
    },
]

print(f"Ground truth loaded: {len(GROUND_TRUTH)} invoices")
for g in GROUND_TRUTH:
    print(f"  {g['vendor_name']:<45} invoice={g['invoice_number']}")

Ground truth loaded: 9 invoices
  Aviko B.V.                                    invoice=93184726
  Nemco Food Trading Inc                        invoice=93828
  Quirch Foods LLC                              invoice=160196
  BARON PHILIPPE DE ROTHSCHILD, S.A.            invoice=2304001154
  BARON PHILIPPE DE ROTHSCHILD MAIΡΟ CHILE S.A  invoice=2002075457
  de Ladoucette                                 invoice=0033043657
  LES VINS GEORGES DUBOEUF                      invoice=2023/5275
  HOBNOB CELLARS                                invoice=2023/5274
  MINUTY S.A.S.                                 invoice=00568057


## 6  Two schemas

In [6]:
HEADER_SCHEMA = {
    "vendor_name":              "string",
    "vendor_address":           "string or null",
    "invoice_number":           "string",
    "purchase_order_number":    "string or null",
    "invoice_date":             "string (YYYY-MM-DD)",
    "sub_total":                "number or null",
    "tax":                      "number or null",
    "total":                    "number",
    "shipping_handling_charge": "number or null",
    "insurance_charge":         "number or null",
    "discount":                 "number or null",
    "ship_to":                  "string or null",
    "bill_to":                  "string or null",
    "delivery_note_number":     "string or null",
    "iban":                     "string or null",
    "vendor_phone":             "string or null",
}
HEADER_SCHEMA_STR = json.dumps(HEADER_SCHEMA, indent=2)

# ── Full schema: header + line items extracted in one LLM call ────────────────
FULL_SCHEMA = dict(HEADER_SCHEMA)
FULL_SCHEMA["line_items"] = [
    {
        "description":          "string ? full product description including pack/size qualifiers needed to distinguish the SKU",
        "quantity":             "number — individual unit count (bottles/pieces/kg, NOT cases)",
        "unit_price":           "number — price per individual unit",
        "line_amount":          "number — row total (quantity × unit_price)",
        "ean_code":             "string or null",
        "material_number":      "string or null",
        "po_number":            "string or null",
        "delivery_note_number": "string or null",
    }
]
FULL_SCHEMA_STR = json.dumps(FULL_SCHEMA, indent=2)

LINE_ITEM_SCHEMA = [
    {
        "description":          "string ? full product description including pack/size qualifiers needed to distinguish the SKU",
        "quantity":             "number ? individual unit count (bottles/pieces/kg, NOT cases)",
        "unit_price":           "number ? price per individual unit",
        "line_amount":          "number ? row total (quantity ? unit_price)",
        "ean_code":             "string or null",
        "material_number":      "string or null",
        "po_number":            "string or null",
        "delivery_note_number": "string or null",
    }
]
LINE_ITEM_SCHEMA_STR = json.dumps(LINE_ITEM_SCHEMA, indent=2)

LINE_ITEM_FIELDS  = ["ean_code", "material_number", "description",
                     "quantity", "unit_price", "line_amount",
                     "po_number", "delivery_note_number"]
LINE_ITEM_NUMERIC = {"quantity", "unit_price", "line_amount"}
print("Schemas ready.")


Schemas ready.


## 7  European number normaliser

In [7]:
_EU_NUMBER_RE = re.compile(r'\b(\d{1,3}(?:\.\d{3})+,\d{1,4})\b')
_US_NUMBER_RE = re.compile(r'\b(\d{1,3}(?:,\d{3})+\.\d{1,4})\b')
_EU_THOU_RE   = re.compile(r'\b(\d{1,3}(?:\.\d{3})+)\b')
# French invoices use a SPACE as the thousands separator: "1 464,96" or "1 464.96"
# This pattern requires a decimal part (,NN or .NN) to avoid false-positives on
# sequences like page numbers, addresses, etc.
_FR_SPACE_THOU_RE = re.compile(r'(?<!\d)(\d{1,3})\s(\d{3})([,.]\d{1,4})\b')

def _eu_to_float_str(m): return m.group(0).replace('.','').replace(',','.')
def _us_to_float_str(m): return m.group(0).replace(',','')

def normalise_numbers(text: str) -> str:
    # 1. French space-thousands FIRST (must run before EU-dot-thousands)
    #    "1 464,96" → "1464.96"   "33 724,44" → "33724.44"
    def _fr_space(m):
        whole = m.group(1) + m.group(2)
        sep   = m.group(3)
        frac  = sep[1:]
        return f"{whole}.{frac}"
    text = _FR_SPACE_THOU_RE.sub(_fr_space, text)
    # 2. EU dot-thousands+comma-decimal: "1.464,96" → "1464.96"
    text = _EU_NUMBER_RE.sub(_eu_to_float_str, text)
    # 3. US comma-thousands+dot-decimal: "1,464.96" → "1464.96"
    text = _US_NUMBER_RE.sub(_us_to_float_str, text)
    # 4. Bare EU thousands (dot-only): "1.464" → "1464" when len >= 7
    def _eu_thou(m):
        s = m.group(0)
        return s.replace('.','') if len(s) >= 7 else s
    return _EU_THOU_RE.sub(_eu_thou, text)

print("Number normaliser ready.")


Number normaliser ready.


## 8  PDF type detection (fixed CLASS C threshold)

**Key fix for AVIKO:** The previous threshold `avg_chars >= 1500 → always NATIVE-TEXT` was wrong. AVIKO has 2593 chars/page from a Dutch T&C text block, but the actual invoice table is embedded as a raster image — pdfminer cannot read it.

New rule: if `image_coverage > 0.80` AND `avg_chars < 4000` → treat as SCANNED. This correctly routes AVIKO to the OCR path.


In [8]:
def _count_pdfminer_chars(pdf_path: str, max_pages: int = 5) -> float:
    total, pages = 0, 0
    try:
        with open(pdf_path, 'rb') as f:
            for page_layout in extract_pages(f):
                for element in page_layout:
                    if isinstance(element, LTTextBox):
                        total += len(element.get_text())
                pages += 1
                if pages >= max_pages: break
    except Exception as e:
        print(f"    pdfminer probe error: {e}")
        return 0.0
    return total / max(pages, 1)


def _get_image_area_fraction(pdf_path: str, max_pages: int = 5) -> float:
    try:
        doc = pymupdf.open(pdf_path)
        max_ratio = 0.0
        for pn in range(min(len(doc), max_pages)):
            page = doc[pn]
            page_area = page.rect.width * page.rect.height
            if page_area < 1.0: continue
            image_area = 0.0
            try:
                for b in page.get_text('dict', flags=0).get('blocks', []):
                    if b.get('type') == 1:
                        bb = b['bbox']
                        image_area += max(0,bb[2]-bb[0]) * max(0,bb[3]-bb[1])
            except Exception: pass
            if image_area < 1.0 and page.get_images(full=False):
                image_area = page_area * 0.90
            max_ratio = max(max_ratio, min(image_area / page_area, 1.0))
        doc.close()
        return max_ratio
    except Exception as e:
        print(f"    Image-fraction probe error: {e}")
        return 0.0




def classify_pdf_mode(pdf_path: str):
    """
    Returns (mode, avg_chars, img_fraction)
      mode in {"native", "scanned", "hybrid"}

    hybrid = significant image coverage + meaningful text layer
    (typical raster invoice body with embedded selectable overlays/headers).
    """
    avg_chars = _count_pdfminer_chars(pdf_path)
    img_fraction = _get_image_area_fraction(pdf_path)

    # No text layer -> fully scanned.
    if avg_chars < 10:
        print(f"    PDF type: SCANNED (avg {avg_chars:.0f} chars/page ? no text layer)")
        return 'scanned', avg_chars, img_fraction

    # Very high text density is typically native text despite embedded images.
    if avg_chars >= 5000:
        print(f"    PDF type: NATIVE-TEXT (avg {avg_chars:.0f} chars/page, image={img_fraction:.0%})")
        return 'native', avg_chars, img_fraction

    # Dominant raster + moderate text -> hybrid.
    if img_fraction > 0.80 and 400 <= avg_chars < 5000:
        print(f"    PDF type: HYBRID (image={img_fraction:.0%} and avg {avg_chars:.0f} chars/page ? mixed raster/text)")
        return 'hybrid', avg_chars, img_fraction

    # Mostly raster with weak text -> scanned.
    if avg_chars < NATIVE_TEXT_CHAR_THRESHOLD:
        print(f"    PDF type: SCANNED (avg {avg_chars:.0f} chars/page, image={img_fraction:.0%})")
        return 'scanned', avg_chars, img_fraction

    if img_fraction > 0.80 and avg_chars < 4000:
        print(f"    PDF type: SCANNED (image={img_fraction:.0%} > 80% and avg {avg_chars:.0f} chars/page < 4000 ? CLASS C: raster with text overlay)")
        return 'scanned', avg_chars, img_fraction

    if img_fraction > 0.60 and avg_chars < 1500:
        print(f"    PDF type: HYBRID (image={img_fraction:.0%} > 60% with overlay text avg {avg_chars:.0f} chars/page)")
        return 'hybrid', avg_chars, img_fraction

    print(f"    PDF type: NATIVE-TEXT (avg {avg_chars:.0f} chars/page, image={img_fraction:.0%})")
    return 'native', avg_chars, img_fraction


def is_scanned_pdf(pdf_path: str) -> bool:
    mode, _, _ = classify_pdf_mode(pdf_path)
    return mode != 'native'


print("PDF type detection ready.")




PDF type detection ready.


## 9  Deskew pre-processing

In [9]:
def _rotate_pdf_pages(src_path: str, dest_path: str) -> bool:
    doc = pymupdf.open(src_path)
    corrected = False
    for page in doc:
        if page.rotation != 0:
            print(f"    Page {page.number + 1}: rotation={page.rotation} deg - correcting")
            page.set_rotation(0)
            corrected = True
    doc.save(dest_path)
    doc.close()
    return corrected


def _deskew_page_image(img_array: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    gray  = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    angle = determine_skew(gray)
    if angle is None or abs(angle) < threshold:
        return img_array
    print(f"    Fine skew detected: {angle:.2f} deg - correcting")
    h, w  = img_array.shape[:2]
    cx, cy = w // 2, h // 2
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)
    M[0, 2] += (new_w - w) / 2
    M[1, 2] += (new_h - h) / 2
    return cv2.warpAffine(img_array, M, (new_w, new_h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT,
                          borderValue=(255, 255, 255))


def _fine_deskew_pdf(src_path: str, dest_path: str, dpi: int = 200) -> bool:
    src_doc  = pymupdf.open(src_path)
    dest_doc = pymupdf.open()
    corrected = False
    for page in src_doc:
        mat  = pymupdf.Matrix(dpi / 72, dpi / 72)
        pix  = page.get_pixmap(matrix=mat, colorspace=pymupdf.csRGB)
        arr  = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
        deskewed = _deskew_page_image(arr)
        if deskewed.shape != arr.shape:
            corrected = True
        img_h, img_w = deskewed.shape[:2]
        page_w, page_h = img_w * 72 / dpi, img_h * 72 / dpi
        new_page = dest_doc.new_page(width=page_w, height=page_h)
        buf = io.BytesIO()
        Image.fromarray(deskewed).save(buf, format="PNG")
        new_page.insert_image(pymupdf.Rect(0, 0, page_w, page_h), stream=buf.getvalue())
    dest_doc.save(dest_path)
    dest_doc.close()
    src_doc.close()
    return corrected


def preprocess_pdf(pdf_path: str, work_dir: str) -> str:
    basename  = os.path.splitext(os.path.basename(pdf_path))[0]
    step1     = os.path.join(work_dir, f"{basename}_rot.pdf")
    rot_fixed = _rotate_pdf_pages(pdf_path, step1)
    after_rot = step1 if rot_fixed else pdf_path
    step2      = os.path.join(work_dir, f"{basename}_deskew.pdf")
    skew_fixed = _fine_deskew_pdf(after_rot, step2)
    after_skew = step2 if skew_fixed else after_rot
    status = []
    if rot_fixed:  status.append("rotation corrected")
    if skew_fixed: status.append("fine skew corrected")
    print(f"    Pre-process: {', '.join(status) if status else 'no corrections needed'}")
    return after_skew


print("Deskew functions defined.")


Deskew functions defined.


## 10  Shared rasterise helper

In [10]:
def _rasterise_page(pdf_path: str, page_num: int, dpi: int = RASTER_DPI) -> np.ndarray:
    doc  = pymupdf.open(pdf_path)
    page = doc[page_num]
    mat  = pymupdf.Matrix(dpi / 72, dpi / 72)
    pix  = page.get_pixmap(matrix=mat, colorspace=pymupdf.csRGB)
    arr  = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    doc.close()
    return arr

print("_rasterise_page defined.")


_rasterise_page defined.


## 11  Load TATR

In [11]:
print("Loading TATR (table-transformer-structure-recognition)...")
TATR_PROCESSOR = DetrImageProcessor.from_pretrained(
    "microsoft/table-transformer-structure-recognition"
)
TATR_MODEL = TableTransformerForObjectDetection.from_pretrained(
    "microsoft/table-transformer-structure-recognition"
)
TATR_MODEL.eval()
if torch.cuda.is_available():
    TATR_MODEL = TATR_MODEL.cuda()
print("TATR ready.")
TATR_ID2LABEL = TATR_MODEL.config.id2label
print("Labels:", TATR_ID2LABEL)


Loading TATR (table-transformer-structure-recognition)...


preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-structure-recognition
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TATR ready.
Labels: {0: 'table', 1: 'table column', 2: 'table row', 3: 'table column header', 4: 'table projected row header', 5: 'table spanning cell'}


## 12  pdfplumber line item extraction (native-text PDFs)

In [12]:
def _match_col_headers(headers):
    """Map column index to schema field. One-field-per-column, one-column-per-field."""
    mapping = {}
    assigned_fields = set()
    for ci, h in enumerate(headers):
        if h is None: continue
        h_lo = str(h).lower().strip()
        for field, aliases in COLUMN_ALIASES.items():
            if field in assigned_fields: continue
            if any(alias in h_lo for alias in aliases):
                mapping[ci] = field
                assigned_fields.add(field)
                break
    return mapping


def _parse_number(s):
    if s is None: return None
    s = str(s).strip()
    if not s: return None
    s = normalise_numbers(s).replace(',', '')
    try: return float(s)
    except ValueError: return None


def _has_numeric_col(rows, mapping):
    numeric_cols = [ci for ci, f in mapping.items() if f in LINE_ITEM_NUMERIC]
    if not numeric_cols: return False
    for row in rows[1:6]:
        for ci in numeric_cols:
            if ci < len(row) and _parse_number(str(row[ci] or '')) is not None:
                return True
    return False


_SUBTOTAL_KW = [
    "subtotal", "sub total", "sub-total", "total", "vat", "tva", "tax",
    "freight", "fret", "insurance", "assurance", "shipping", "handling",
    "sea freight", "ocean freight", "air freight", "balance due",
    "amount due", "net amount", "discount", "escompte",
]


def _is_subtotal_row(item):
    desc = str(item.get('description') or '').lower()
    has_up  = item.get('unit_price')  is not None
    has_qty = item.get('quantity')    is not None
    if not has_up and not has_qty and any(kw in desc for kw in _SUBTOTAL_KW):
        return True
    return desc.strip() in _SUBTOTAL_KW


def _rows_to_line_items(rows, mapping, start_row=1):
    items = []
    for row in rows[start_row:]:
        item = {f: None for f in LINE_ITEM_FIELDS}
        has_desc = False
        for ci, field in mapping.items():
            if ci >= len(row): continue
            val = str(row[ci] or '').strip()
            if not val: continue
            if field in LINE_ITEM_NUMERIC:
                item[field] = _parse_number(val)
            else:
                item[field] = val
                if field == 'description': has_desc = True
        has_numeric = any(item.get(f) is not None for f in LINE_ITEM_NUMERIC)
        if has_desc or has_numeric:
            items.append(item)
    return items


def pdfplumber_extract_line_items(pdf_path, page_nums):
    """
    Extract line items using pdfplumber with three-strategy cascade.
    Returns (items, found_any_table) so caller can decide on OCR fallback.
    """
    all_items = []
    found_any = False
    _STRATEGIES = [
        {},
        {"vertical_strategy": "text", "horizontal_strategy": "text"},
        {"vertical_strategy": "text", "horizontal_strategy": "text",
         "snap_tolerance": 5, "join_tolerance": 5},
    ]
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for pn in page_nums:
                if pn >= len(pdf.pages): continue
                page = pdf.pages[pn]
                for strategy in _STRATEGIES:
                    try:
                        tables = page.extract_tables(strategy) if strategy else page.extract_tables()
                    except Exception: continue
                    if not tables: continue
                    best_table, best_score, best_mapping = None, -1, {}
                    for tbl in tables:
                        if not tbl or len(tbl) < 2: continue
                        mapping = _match_col_headers([str(c or '') for c in tbl[0]])
                        score   = len(mapping)
                        if score > best_score and _has_numeric_col(tbl, mapping):
                            best_score, best_table, best_mapping = score, tbl, mapping
                    if best_table is not None and best_score > 0:
                        sname = ['std','text','loose'][_STRATEGIES.index(strategy)]
                        print(f"    pdfplumber p{pn+1} ({sname}): score={best_score} mapping={best_mapping}")
                        items = _rows_to_line_items(best_table, best_mapping)
                        all_items.extend(it for it in items if not _is_subtotal_row(it))
                        found_any = True
                        break
                else:
                    print(f"    pdfplumber: no scoreable table on page {pn+1}")
    except Exception as e:
        print(f"    pdfplumber error: {e}")
    print(f"    pdfplumber: {len(all_items)} line items (found_any={found_any})")
    return all_items, found_any


print("pdfplumber extractor ready.")



def _table_df_to_rows(df):
    if df is None or getattr(df, 'empty', True):
        return []
    rows = []
    cols = [str(c or '').strip() for c in list(df.columns)]
    if any(c and c.lower() != 'nan' for c in cols):
        rows.append(cols)
    for _, row in df.fillna('').iterrows():
        rows.append([str(v or '').strip() for v in row.tolist()])
    return rows


def _score_rows_as_table(rows):
    if not rows or len(rows) < 2:
        return -1, {}, []
    best_score, best_mapping, best_rows = -1, {}, []
    for hdr_idx in range(min(5, len(rows))):
        hdr = [str(c or '') for c in rows[hdr_idx]]
        mapping = _match_col_headers(hdr)
        data_rows = rows[hdr_idx:]
        score = len(mapping)

        # Prefer mappings that actually cover core numeric/item fields.
        if mapping and _mapping_is_usable(mapping) and _has_numeric_col(data_rows, mapping):
            score += 8
        elif len(rows[0]) >= 4:
            mapping = _positional_column_mapping(
                rows,
                len(rows[0]),
                row_start=max(hdr_idx + 1, 1),
                row_end=min(len(rows), hdr_idx + 12),
            )
            score = len(mapping)
            if mapping and _mapping_is_usable(mapping):
                score += 3

        if score > best_score and mapping and _mapping_is_usable(mapping):
            best_score, best_mapping, best_rows = score, mapping, data_rows
    return best_score, best_mapping, best_rows


def camelot_extract_line_items(pdf_path, page_nums):
    if camelot is None:
        return [], False
    pages = ','.join(str(p + 1) for p in page_nums)
    all_items = []
    found_any = False
    try:
        tables = camelot.read_pdf(pdf_path, pages=pages, flavor='stream')
    except Exception as e:
        print(f'    camelot error: {e}')
        return [], False

    for i, table in enumerate(tables):
        try:
            rows = _table_df_to_rows(table.df)
        except Exception:
            continue
        if not rows:
            continue
        found_any = True
        score, mapping, data_rows = _score_rows_as_table(rows)
        if score <= 0 or not mapping:
            continue
        print(f'    camelot table {i+1}: score={score} mapping={mapping}')
        items = _rows_to_line_items(data_rows, mapping, start_row=1)
        items = [it for it in items if not _is_subtotal_row(it)]
        if items:
            all_items.extend(items)
    print(f'    camelot: {len(all_items)} line items (found_any={found_any})')
    return all_items, found_any


def tabula_extract_line_items(pdf_path, page_nums):
    if tabula is None:
        return [], False
    pages = [p + 1 for p in page_nums]
    all_items = []
    found_any = False
    try:
        dfs = tabula.read_pdf(pdf_path, pages=pages, multiple_tables=True, stream=True, guess=True, pandas_options={"dtype": str})
    except Exception as e:
        print(f'    tabula error: {e}')
        return [], False
    for i, df in enumerate(dfs or []):
        rows = _table_df_to_rows(df)
        if not rows:
            continue
        found_any = True
        score, mapping, data_rows = _score_rows_as_table(rows)
        if score <= 0 or not mapping:
            continue
        print(f'    tabula table {i+1}: score={score} mapping={mapping}')
        items = _rows_to_line_items(data_rows, mapping, start_row=1)
        items = [it for it in items if not _is_subtotal_row(it)]
        if items:
            all_items.extend(items)
    print(f'    tabula: {len(all_items)} line items (found_any={found_any})')
    return all_items, found_any


pdfplumber extractor ready.


## 13  TATR cell-level line item extraction (scanned PDFs)

**Root cause of 0 line items in v7.2:** `_ocr_cell` with `--psm 7` on isolated
~15px tall cell crops produces empty strings. Both alias matching and the positional
fallback then see empty columns and give up.

**Fix:** Run `image_to_data()` on the **full page** (Tesseract sees full context,
produces good results). Then assign each recognised word to the TATR cell whose
bounding box contains the word's centroid. Words with confidence < 30 are discarded.
This is the intended usage pattern for TATR + an OCR engine.

In [13]:
def _mapping_is_usable(mapping):
    """True if mapping covers at least 2 of the 4 key line-item fields."""
    key_fields = {'description', 'quantity', 'unit_price', 'line_amount'}
    return len(key_fields.intersection(mapping.values())) >= 2


def _positional_column_mapping(grid, col_count, row_start=0, row_end=None):
    """
    Infer column roles by position when header alias matching fails.

    FIX v8.1: Scans ALL data rows (not just rows 1-7) because the product
    table is typically buried at row ~35+ in a full-page 79-row TATR grid.

    Rightmost numeric col  -> line_amount
    Next rightmost numeric -> unit_price
    Small-valued numeric   -> quantity
    Longest text col       -> description
    """
    if col_count == 0 or len(grid) < 2:
        return {}
    # Scan all rows from row_start to row_end (default: entire grid)
    end = row_end if row_end is not None else len(grid)
    sample_rows = grid[row_start:end]
    if not sample_rows:
        return {}

    col_numeric_vals = [[] for _ in range(col_count)]
    col_text_lens    = [[] for _ in range(col_count)]
    for row in sample_rows:
        for ci in range(min(col_count, len(row))):
            v = _parse_number(str(row[ci] or ''))
            if v is not None:
                col_numeric_vals[ci].append(v)
            else:
                col_text_lens[ci].append(len(str(row[ci] or '')))

    n_rows = len(sample_rows)
    # A column is "numeric" if ≥25% of its rows have parseable numbers
    # (lowered from 40% because sparse invoice tables have many empty cells)
    numeric_cols = [ci for ci in range(col_count)
                    if len(col_numeric_vals[ci]) >= max(1, n_rows * 0.25)]
    if len(numeric_cols) < 2:
        print(f'    Positional: not enough numeric cols ({len(numeric_cols)}/2 needed, '
              f'scanned {n_rows} rows)')
        return {}

    mapping = {}
    nc_sorted = sorted(numeric_cols, reverse=True)
    mapping[nc_sorted[0]] = 'line_amount'
    if len(nc_sorted) >= 2:
        mapping[nc_sorted[1]] = 'unit_price'
    for ci in sorted(numeric_cols):
        if ci in mapping:
            continue
        if not col_numeric_vals[ci]:
            continue
        avg_val = sum(col_numeric_vals[ci]) / len(col_numeric_vals[ci])
        if avg_val < 10000:
            mapping[ci] = 'quantity'
            break
    text_cols = [
        (ci, sum(col_text_lens[ci]) / max(len(col_text_lens[ci]), 1))
        for ci in range(col_count)
        if ci not in mapping and col_text_lens[ci]
    ]
    if text_cols:
        mapping[max(text_cols, key=lambda x: x[1])[0]] = 'description'
    print(f'    Positional col mapping: {mapping}')
    return mapping


def _tatr_get_rows_and_cols(img_pil):
    """Run TATR and return (row_boxes, col_boxes) sorted by y0 / x0."""
    inputs = TATR_PROCESSOR(images=img_pil, return_tensors='pt')
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = TATR_MODEL(**inputs)
    target_sizes = torch.tensor([img_pil.size[::-1]])
    results = TATR_PROCESSOR.post_process_object_detection(
        outputs, threshold=TATR_CONFIDENCE, target_sizes=target_sizes
    )[0]
    rows, cols = [], []
    for score, label_id, box in zip(
        results['scores'], results['labels'], results['boxes']
    ):
        label = TATR_MODEL.config.id2label[label_id.item()]
        x0, y0, x1, y1 = box.tolist()
        entry = {'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1, 'score': score.item()}
        if 'row'  in label.lower(): rows.append(entry)
        elif 'col' in label.lower(): cols.append(entry)
    rows.sort(key=lambda r: r['y0'])
    cols.sort(key=lambda c: c['x0'])
    return rows, cols


def _build_grid_from_words(arr, rows, cols):
    """
    Full-page OCR -> assign each word to a TATR (row, col) cell by centroid.

    Runs image_to_data() on the full page so Tesseract has full document
    context. Each recognised word (conf >= 30) is placed into the cell
    whose (row y-span, col x-span) contains the word's centroid.
    """
    img_pil = Image.fromarray(arr)
    data = pytesseract.image_to_data(
        img_pil,
        config='--oem 1 --psm 6',
        output_type=pytesseract.Output.DICT,
    )

    n_rows = len(rows)
    n_cols = len(cols)
    grid   = [['' for _ in range(n_cols)] for _ in range(n_rows)]

    for i in range(len(data['text'])):
        word = data['text'][i].strip()
        conf = int(data['conf'][i])
        if not word or conf < 30:
            continue

        wx = data['left'][i]
        wy = data['top'][i]
        ww = data['width'][i]
        wh = data['height'][i]
        cx = wx + ww / 2
        cy = wy + wh / 2

        # Find TATR row containing cy
        ri = None
        for r_idx, row in enumerate(rows):
            if row['y0'] <= cy <= row['y1']:
                ri = r_idx
                break
        if ri is None:
            dists = [min(abs(cy - r['y0']), abs(cy - r['y1'])) for r in rows]
            ri = int(min(range(n_rows), key=lambda i: dists[i]))

        # Find TATR col containing cx
        ci = None
        for c_idx, col in enumerate(cols):
            if col['x0'] <= cx <= col['x1']:
                ci = c_idx
                break
        if ci is None:
            dists = [min(abs(cx - c['x0']), abs(cx - c['x1'])) for c in cols]
            ci = int(min(range(n_cols), key=lambda i: dists[i]))

        sep = ' ' if grid[ri][ci] else ''
        grid[ri][ci] = grid[ri][ci] + sep + word

    return grid


def _find_best_table_region(grid, col_count):
    """
    Scan ALL rows to find the contiguous region that looks most like a
    product line-item table:
      - has a header row (alias match score >= 2), OR
      - has the most numeric content across multiple columns.

    Returns (header_row_idx, mapping, data_start_row, data_end_row).

    FIX v8.1: Previously only scanned rows 0-5 for the header.
    For a 79-row full-page grid, the product table header sits at row ~35.
    Now scans every row.
    """
    n_rows = len(grid)

    # ── Pass 1: find header row by alias matching ──────────────────────────
    best_alias_score, best_alias_idx, best_alias_map = 0, -1, {}
    for ri in range(n_rows):
        m = _match_col_headers(grid[ri])
        s = len(m)
        if s > best_alias_score:
            best_alias_score, best_alias_idx, best_alias_map = s, ri, m

    if _mapping_is_usable(best_alias_map):
        print(f'    Table region: header at row {best_alias_idx} '
              f'(alias score={best_alias_score}) mapping={best_alias_map}')
        return best_alias_idx, best_alias_map, best_alias_idx + 1, n_rows

    # ── Pass 2: find densest numeric region by sliding window ─────────────
    # Score each row: how many columns contain a parseable number?
    row_numeric_scores = []
    for row in grid:
        score = sum(
            1 for ci in range(min(col_count, len(row)))
            if _parse_number(str(row[ci] or '')) is not None
        )
        row_numeric_scores.append(score)

    # Find the window of consecutive rows with highest total numeric density
    # Use a window of min(20, n_rows//3) rows
    window = max(5, min(20, n_rows // 3))
    best_window_score, best_window_start = -1, 1
    for start in range(1, max(2, n_rows - window + 1)):
        s = sum(row_numeric_scores[start:start + window])
        if s > best_window_score:
            best_window_score, best_window_start = s, start

    if best_window_score == 0:
        print(f'    Table region: no numeric content found in any window')
        return -1, {}, 1, n_rows

    data_start = best_window_start
    data_end   = min(n_rows, best_window_start + window + 5)

    print(f'    Table region: numeric window rows {data_start}-{data_end} '
          f'(score={best_window_score}), alias scan failed ({best_alias_map})')

    # Try positional mapping on just the numeric window
    mapping = _positional_column_mapping(grid, col_count,
                                          row_start=data_start, row_end=data_end)
    return -1, mapping, data_start, data_end


def tatr_extract_line_items_from_page(pdf_path, page_num):
    """
    Cell-level TATR extraction using full-page OCR word assignment.

    Steps:
      1. Rasterise page at RASTER_DPI
      2. TATR -> row + column bboxes
      3. image_to_data() on full page -> assign words to cells by centroid
      4. _find_best_table_region() scans ALL rows for header + numeric content
      5. Extract line items from the identified data region
    """
    arr     = _rasterise_page(pdf_path, page_num)
    img_pil = Image.fromarray(arr)
    rows, cols = _tatr_get_rows_and_cols(img_pil)

    if not rows or not cols:
        print(f'    TATR p{page_num+1}: no table detected '
              f'(rows={len(rows)}, cols={len(cols)})')
        return []

    print(f'    TATR p{page_num+1}: {len(rows)} rows x {len(cols)} cols '
          f'— building grid via word assignment')
    grid = _build_grid_from_words(arr, rows, cols)
    if not grid:
        return []

    # Print first few non-empty rows for diagnostics
    non_empty = [(ri, r) for ri, r in enumerate(grid) if any(c.strip() for c in r)]
    print(f'    Grid has {len(non_empty)} non-empty rows out of {len(grid)}')
    for ri, row in non_empty[:5]:
        print(f'      row {ri:3d}: {row}')
    if len(non_empty) > 5:
        print(f'      ... ({len(non_empty)-5} more non-empty rows)')

    header_idx, mapping, data_start, data_end = _find_best_table_region(
        grid, len(cols)
    )

    if not mapping:
        print(f'    TATR p{page_num+1}: could not map any columns — skipping')
        return []

    items = _rows_to_line_items(grid, mapping, start_row=data_start)
    # Only keep rows up to data_end
    items_in_window = []
    for ri, row in enumerate(grid[data_start:data_end], start=data_start):
        item = {f: None for f in LINE_ITEM_FIELDS}
        has_desc, has_numeric = False, False
        for ci, field in mapping.items():
            if ci >= len(row):
                continue
            val = str(row[ci] or '').strip()
            if not val:
                continue
            if field in LINE_ITEM_NUMERIC:
                parsed = _parse_number(normalise_numbers(val))
                if parsed is not None:
                    item[field] = parsed
                    has_numeric = True
            else:
                item[field] = val
                if field == 'description':
                    has_desc = True
        if (has_desc or has_numeric) and not _is_subtotal_row(item):
            items_in_window.append(item)

    print(f'    TATR p{page_num+1}: {len(items_in_window)} line items extracted')
    return items_in_window


def tatr_extract_line_items(pdf_path, page_nums):
    """Run TATR word-assignment extraction across all invoice pages."""
    all_items = []
    for pn in page_nums:
        all_items.extend(tatr_extract_line_items_from_page(pdf_path, pn))
    return all_items


print('TATR word-assignment extractor ready (v8.1 — full-page row scan).')


_TABLE_UNIT_RE = r'(?:lb|lbs|kg|g|gr|ml|cl|l|pcs|pc|btl|btl\.|bottles?|cs|case|cases|unit|units)'


def _ocr_text_from_array(arr, psm=6):
    if arr is None or getattr(arr, 'size', 0) == 0:
        return ''
    img_pil = Image.fromarray(arr)
    text = pytesseract.image_to_string(img_pil, config=f'--oem 1 --psm {psm}')
    return re.sub(r'\n{3,}', '\n\n', text or '').strip()


def _table_text_score(text):
    text = str(text or '')
    if not text.strip():
        return -1
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if len(lines) < 3:
        return -1
    numeric = sum(1 for ln in lines if len(re.findall(r'\d', ln)) >= 3)
    moneyish = sum(1 for ln in lines if re.search(r'\d[.,]\d{2}', ln))
    skuish = sum(1 for ln in lines if re.search(r'(?i)(pack|case|cs|btl|bottle|ml|lbs?|kg|g|igp|a\.?o\.?p\.?|shrimp|fries|hashbrown|coconut|rose|rouge|chardonnay|pinot|minuty|ladoucette|escudo)', ln))
    return numeric * 2 + moneyish * 2 + skuish


def extract_table_text_from_page(pdf_path, page_num):
    arr = _rasterise_page(pdf_path, page_num, dpi=TABLE_RASTER_DPI)
    h, w = arr.shape[:2]
    crops = []

    # Prefer the lower body of the invoice where product tables usually live.
    for y0r, y1r, x0r, x1r in [
        (0.22, 0.96, 0.02, 0.98),
        (0.32, 0.96, 0.02, 0.98),
        (0.42, 0.96, 0.02, 0.98),
        (0.28, 0.92, 0.08, 0.96),
    ]:
        y0, y1 = int(h * y0r), int(h * y1r)
        x0, x1 = int(w * x0r), int(w * x1r)
        crops.append(arr[y0:y1, x0:x1])

    # If TATR found row boxes, use their vertical extent as an extra crop candidate.
    try:
        rows, _ = _tatr_get_rows_and_cols(Image.fromarray(arr))
    except Exception:
        rows = []
    if rows:
        y0 = max(0, int(min(r['y0'] for r in rows) - 20))
        y1 = min(h, int(max(r['y1'] for r in rows) + 20))
        crops.insert(0, arr[y0:y1, :])

    best_text, best_score = '', -1
    for crop in crops:
        for psm in (6, 11):
            text = _ocr_text_from_array(crop, psm=psm)
            score = _table_text_score(text)
            if score > best_score:
                best_text, best_score = text, score

    if best_score >= 6:
        print(f'    Table-text p{page_num+1}: selected OCR crop (score={best_score})')
        return best_text
    print(f'    Table-text p{page_num+1}: no strong OCR crop found (best_score={best_score})')
    return ''


def extract_table_text(pdf_path, page_nums):
    parts = []
    for pn in page_nums:
        text = extract_table_text_from_page(pdf_path, pn)
        if text:
            parts.append(f'## Page {pn+1}\n' + text)
    return '\n\n'.join(parts)


TATR word-assignment extractor ready (v8.1 — full-page row scan).


## 14  Line item arithmetic validator

In [15]:
import re

def _line_item_pack_hint(desc):
    desc = str(desc or '')
    return bool(re.search(
        r'(?i)(\b\d+\s*[x/]\s*\d+\s*(?:g|gr|kg|lb|lbs|ml|cl|l)\b|\breg\s*\d+\b|\bpack\b|\bcase\b|\bcs\b|\bbottle\b|\bbout\.?\b)',
        desc,
    ))

def _strip_leading_code_tokens(text):
    s = str(text or '').strip()
    if not s:
        return s
    toks = s.split()
    noisy = {
        'BOLT.', 'BOLT', 'BOUT.', 'BOUT', 'BTL.', 'BTL', 'BLE', 'CS', 'CASE', 'CASES',
        'PCS', 'PC', 'U', 'UN', 'UNIT', 'UNITS'
    }
    cleaned = []
    dropping = True
    for tok in toks:
        t = tok.strip(' |,;')
        upper = t.upper()
        is_num = bool(re.match(r'^\d+(?:[.,]\d+)?$', t))
        is_code = bool(re.match(r'^[A-Z]{0,3}\d{2,}[A-Z0-9-]*$', upper))
        if dropping and (is_num or is_code or upper in noisy):
            continue
        dropping = False
        cleaned.append(tok)
    out = ' '.join(cleaned).strip()
    return out or s

def _extract_pack_multiplier(desc):
    desc = str(desc or '')
    patterns = [
        r'(?i)\bpack\s*[:=-]?\s*(\d+)\s*x\s*\d+(?:[.,]\d+)?\s*(?:g|gr|kg|lb|lbs|ml|cl|l)\b',
        r'(?i)\b(\d+)\s*x\s*\d+(?:[.,]\d+)?\s*(?:g|gr|kg|lb|lbs|ml|cl|l)\b',
        r'(?i)\b(\d+)\s*/\s*\d+(?:[.,]\d+)?\s*(?:g|gr|kg|lb|lbs|ml|cl|l)\b',
    ]
    for pattern in patterns:
        m = re.search(pattern, desc)
        if m:
            try:
                return float(m.group(1))
            except (TypeError, ValueError):
                return None
    return None

def _clean_line_item_description(desc, item=None):
    s = str(desc or '').strip()
    if not s:
        return s

    s = s.replace('\u201d', '"').replace('\u201c', '"').replace('\u2019', "'")
    s = s.replace('\u201d', '"').replace('\u201c', '"').replace('\u2019', "'")
    s = re.sub(r'(?i)\b1\.?G\.?P\.?\b', 'IGP', s)
    s = _strip_leading_code_tokens(s)
    s = re.sub(r'(?i)^\s*(?:\d+\s+)?(?:BOUT\.?|BTL\.?|BOTTLES?\.?|CS|CASES?)\s+', '', s)
    s = re.sub(r'(?i)^\s*(?:I\.?G\.?P\.?|A\.?O\.?P\.?|A\.?O\.?C\.?)\s*[|:-]\s*', '', s)
    s = re.sub(r'(?i)^\s*cpj\s*-\s*', 'CPJ-', s)
    s = re.sub(r'(?i)\bP[!I]NOT\b', 'PINOT', s)
    s = re.sub(r"\s+([\"'])", r'\1', s)
    s = re.sub(r'\s{2,}', ' ', s).strip()
    s = re.sub(r'(?i)\s+(?:AE|FR|ES)\s*$', '', s).strip()

    if item:
        for key in ('unit_price', 'line_amount'):
            val = item.get(key)
            if val is None:
                continue
            try:
                f = float(val)
            except (TypeError, ValueError):
                continue
            for tok in {f'{f:.2f}', f'{f:.4f}', f'{f:.2f}'.replace('.', ','), f'{f:.4f}'.replace('.', ',')}:
                s = re.sub(rf'(?<!\w){re.escape(tok)}(?!\w)', ' ', s)
        qty = item.get('quantity')
        try:
            q = float(qty) if qty is not None else None
        except (TypeError, ValueError):
            q = None
        if q is not None:
            qty_tokens = {f'{q:.0f}', f'{q:.2f}', f'{q:.2f}'.replace('.', ','), format(int(round(q)), ',')}
            qty_tokens = {t for t in qty_tokens if t and t != '0'}
            if qty_tokens:
                qty_pat = '|'.join(re.escape(t) for t in sorted(qty_tokens, key=len, reverse=True))
                s = re.sub(rf'(?i)\s+(?:{qty_pat})\s*{_TABLE_UNIT_RE}\s*$', '', s).strip()
                s = re.sub(rf'(?i)\s+(?:{qty_pat})\s*$', '', s).strip()

    tail_code = re.search(r'\b(\d{7,10})\s*$', s)
    if tail_code:
        mat = str((item or {}).get('material_number') or '').strip()
        ean = str((item or {}).get('ean_code') or '').strip()
        if tail_code.group(1) not in {mat, ean}:
            s = re.sub(r'\s+\d{7,10}\s*$', '', s).strip()

    # Keep pack-size hints like "4x2500g" / "reg3", but remove trailing alcohol/volume
    # noise that tends to come from OCR on wine invoices.
    s = re.sub(r'(?i)\s+\d+[,.]?\d*\s*[°º\?]?\s+\d+\s*ml[a-z]?\s*$', '', s).strip()
    s = re.sub(r'(?i)\s+\d+[,.]?\d*\s*[°º\?]?\s+\d+\s*ml[a-z]?\s+[A-Z]{0,2}\s*$', '', s).strip()
    s = re.sub(r'(?i)\s*,?\s*alcohol\s+\d+[,.]?\d*\s*%.*$', '', s).strip()
    s = re.sub(r'\s{2,}', ' ', s).strip(' ,;:-')
    return s

def _line_item_desc_score(desc):
    desc = str(desc or '').strip()
    if not desc:
        return 0.0
    tokens = re.findall(r'[A-Za-z0-9"/%.-]+', desc)
    score = min(len(tokens), 14)
    if _line_item_pack_hint(desc):
        score += 4
    if any(ch.isdigit() for ch in desc):
        score += 1
    return float(score)


def _line_item_arithmetic_ok(item):
    qty = item.get('quantity')
    up  = item.get('unit_price')
    la  = item.get('line_amount')
    if qty in (None, 0) or up is None or la is None:
        return False
    try:
        qty, up, la = float(qty), float(up), float(la)
    except (TypeError, ValueError):
        return False
    # Do not treat zero-price/zero-amount promo rows as arithmetic-strong.
    if up <= 0 or la <= 0:
        return False
    tol = max(abs(la) * 0.02, 0.05)
    return abs((qty * up) - la) <= tol

def _match_line_item_candidates(item_a, item_b):
    score = 0.0
    ean_a = str(item_a.get('ean_code') or '').strip()
    ean_b = str(item_b.get('ean_code') or '').strip()
    if ean_a and ean_b and ean_a == ean_b:
        score += 20

    mat_a = str(item_a.get('material_number') or '').strip().lower()
    mat_b = str(item_b.get('material_number') or '').strip().lower()
    if mat_a and mat_b and mat_a == mat_b:
        score += 15

    la_a = item_a.get('line_amount')
    la_b = item_b.get('line_amount')
    try:
        if la_a is not None and la_b is not None and abs(float(la_a) - float(la_b)) <= max(abs(float(la_b)) * 0.02, 0.05):
            score += 12
    except (TypeError, ValueError):
        pass

    qty_a, qty_b = item_a.get('quantity'), item_b.get('quantity')
    up_a, up_b = item_a.get('unit_price'), item_b.get('unit_price')
    try:
        if qty_a is not None and qty_b is not None and abs(float(qty_a) - float(qty_b)) <= max(abs(float(qty_b)) * 0.02, 0.05):
            score += 5
    except (TypeError, ValueError):
        pass
    try:
        if up_a is not None and up_b is not None and abs(float(up_a) - float(up_b)) <= max(abs(float(up_b)) * 0.02, 0.05):
            score += 5
    except (TypeError, ValueError):
        pass

    desc_a = set(re.findall(r'[a-z0-9]+', str(item_a.get('description') or '').lower()))
    desc_b = set(re.findall(r'[a-z0-9]+', str(item_b.get('description') or '').lower()))
    if desc_a and desc_b:
        overlap = len(desc_a & desc_b)
        if overlap:
            score += min(overlap, 6)
    return score

def _merge_line_item_candidates(primary_items, secondary_items):
    if not primary_items:
        return list(secondary_items or [])
    if not secondary_items:
        return list(primary_items or [])

    merged = []
    used_secondary = set()
    for primary in primary_items:
        best_idx, best_score = None, -1.0
        for idx, secondary in enumerate(secondary_items):
            if idx in used_secondary:
                continue
            score = _match_line_item_candidates(primary, secondary)
            if score > best_score:
                best_idx, best_score = idx, score

        item = dict(primary)
        if best_idx is not None and best_score >= 12:
            secondary = secondary_items[best_idx]
            used_secondary.add(best_idx)
            for field in ('ean_code', 'material_number', 'po_number', 'delivery_note_number'):
                if not item.get(field) and secondary.get(field):
                    item[field] = secondary[field]
            if _line_item_desc_score(secondary.get('description')) > _line_item_desc_score(item.get('description')):
                item['description'] = secondary.get('description')
            for field in ('quantity', 'unit_price', 'line_amount'):
                if item.get(field) in (None, 0, 0.0) and secondary.get(field) not in (None, 0, 0.0):
                    item[field] = secondary[field]
        merged.append(item)
    return merged




def _table_items_look_viable(items, header=None):
    items = list(items or [])
    if not items:
        return False

    zero_rows = 0
    positive_rows = 0
    vals = []
    for it in items:
        up = _safe_float(it.get('unit_price'))
        la = _safe_float(it.get('line_amount'))
        if la in (None, 0.0) or up in (None, 0.0):
            zero_rows += 1
        if la not in (None, 0.0):
            positive_rows += 1
            vals.append(float(la))

    # Allow mixed tables with helper/zero rows as long as we still have useful positive lines.
    if zero_rows > max(2, int(len(items) * 0.60)) and positive_rows < 2:
        return False
    if positive_rows < max(1, len(items) // 3):
        return False
    if not vals:
        return False

    if header and header.get('sub_total') is not None:
        try:
            st = float(header.get('sub_total'))
            ratio = sum(vals) / max(abs(st), 1.0)
            if ratio < 0.45 or ratio > 1.55:
                return False
        except Exception:
            pass

    if header and header.get('total') is not None:
        try:
            tt = float(header.get('total'))
            ratio_t = sum(vals) / max(abs(tt), 1.0)
            if ratio_t < 0.30 or ratio_t > 1.70:
                return False
        except Exception:
            pass
    return True

def _llm_items_are_strong(items, header=None):
    items = list(items or [])
    if not items:
        return False
    arithmetic_ok = sum(1 for item in items if _line_item_arithmetic_ok(item))
    if arithmetic_ok / max(len(items), 1) < 0.8:
        return False

    # Don't call LLM items "strong" when descriptions are too short/generic.
    avg_desc = sum(_line_item_desc_score(item.get('description')) for item in items) / max(len(items), 1)
    if avg_desc < 6.5:
        return False

    vals = [_safe_float(it.get('line_amount')) for it in items]
    vals = [v for v in vals if v is not None]

    if header and header.get('sub_total') is not None and vals:
        try:
            st = float(header.get('sub_total'))
            ratio = sum(vals) / max(abs(st), 1.0)
            if ratio < 0.6 or ratio > 1.4:
                return False
        except Exception:
            pass

    # Extra guard: table sums wildly below/above total are almost always OCR junk.
    if header and header.get('total') is not None and vals:
        try:
            tt = float(header.get('total'))
            ratio_t = sum(vals) / max(abs(tt), 1.0)
            if ratio_t < 0.35 or ratio_t > 1.5:
                return False
        except Exception:
            pass
    return True

def _line_item_set_score(items, header=None):
    if not items:
        return -1000000.0, 'empty'

    arithmetic_ok = sum(1 for item in items if _line_item_arithmetic_ok(item))
    desc_score = sum(_line_item_desc_score(item.get('description')) for item in items) / len(items)
    id_score = sum(
        1 for item in items
        if item.get('ean_code') or item.get('material_number') or item.get('po_number')
    ) / len(items)

    amount_values = []
    zero_rows = 0
    for item in items:
        q = _safe_float(item.get('quantity'))
        up = _safe_float(item.get('unit_price'))
        la = _safe_float(item.get('line_amount'))
        if la is not None:
            amount_values.append(la)
        if la in (None, 0.0) or up in (None, 0.0) or q in (None, 0.0):
            zero_rows += 1

    subtotal_bonus = 0.0
    subtotal_penalty = 0.0
    if header and header.get('sub_total') is not None and amount_values:
        try:
            subtotal = float(header['sub_total'])
            predicted = sum(amount_values)
            delta = abs(predicted - subtotal)
            rel = delta / max(abs(subtotal), 1.0)
            subtotal_bonus = max(0.0, 8.0 - (rel * 40.0))
            if rel > 0.35:
                subtotal_penalty = min(20.0, rel * 15.0)
        except (TypeError, ValueError):
            subtotal_bonus = 0.0
            subtotal_penalty = 0.0

    zero_penalty = zero_rows * 3.0
    # For ID-rich table candidates, allow helper/zero rows without killing the set.
    if len(items) >= 4 and id_score >= 0.5:
        zero_penalty = zero_rows * 0.6

    score = (
        arithmetic_ok * 4.0 +
        desc_score +
        (id_score * 3.0) +
        min(len(items), 12) * 0.5 +
        subtotal_bonus -
        subtotal_penalty -
        zero_penalty
    )
    diag = (
        f'count={len(items)} arithmetic={arithmetic_ok}/{len(items)} '
        f'desc={desc_score:.1f} ids={id_score:.2f} subtotal_bonus={subtotal_bonus:.2f} '
        f'subtotal_penalty={subtotal_penalty:.2f} zero_penalty={zero_penalty:.2f}'
    )
    return score, diag





def _choose_line_item_set(llm_items, structural_items, header=None, table_items=None):
    candidates = [
        ('llm', list(llm_items or [])),
        ('llm+struct', _merge_line_item_candidates(llm_items or [], structural_items or [])),
        ('struct', list(structural_items or [])),
        ('struct+llm', _merge_line_item_candidates(structural_items or [], llm_items or [])),
    ]

    llm_strong = _llm_items_are_strong(llm_items, header)
    table_ok = _table_items_look_viable(table_items, header)
    llm_arith = 0.0
    if llm_items:
        llm_arith = sum(1 for it in llm_items if _line_item_arithmetic_ok(it)) / max(len(llm_items), 1)

    allow_table = (not llm_items) or (llm_arith <= 0.65) or (len(llm_items) <= 2 and not llm_strong)
    if table_items and table_ok and allow_table:
        candidates.extend([
            ('table', list(table_items or [])),
            ('table+llm', _merge_line_item_candidates(table_items or [], llm_items or [])),
            ('table+struct', _merge_line_item_candidates(table_items or [], structural_items or [])),
            ('llm+table', _merge_line_item_candidates(llm_items or [], table_items or [])),
        ])
    elif table_items and not table_ok:
        print('        Table candidates ignored: failed viability checks.')
    elif table_items and not allow_table:
        print('        Table candidates skipped: LLM line items already preferred.')

    best_name, best_items, best_score = 'llm', [], -1000000.0
    for name, items in candidates:
        score, diag = _line_item_set_score(items, header)
        print(f'        Candidate {name:<10} score={score:.2f} :: {diag}')
        if score > best_score:
            best_name, best_items, best_score = name, items, score

    return best_items, best_name, best_score

def _is_noise_line_item(item):
    desc = _clean_line_item_description(item.get('description'), item)
    qty = item.get('quantity')
    up = item.get('unit_price')
    la = item.get('line_amount')
    try:
        qty = float(qty) if qty is not None else None
    except (TypeError, ValueError):
        qty = None
    try:
        up = float(up) if up is not None else None
    except (TypeError, ValueError):
        up = None
    try:
        la = float(la) if la is not None else None
    except (TypeError, ValueError):
        la = None

    if not desc:
        return True

    noise_kw = re.compile(
        r'(?i)\b(?:marinated with|water[, ]+citric|sodium tripolyphosphate|terms and conditions|general conditions|all contracts)\b'
    )
    if noise_kw.search(desc) and not item.get('ean_code') and not item.get('material_number'):
        return True

    if la in (None, 0.0) and not item.get('ean_code') and not item.get('material_number'):
        return True

    if up in (None, 0.0) and qty and qty >= 500 and not item.get('ean_code') and not item.get('material_number'):
        return True

    if (qty in (None, 0.0) or la in (None, 0.0)) and not item.get('ean_code') and not item.get('material_number'):
        if len(desc.split()) <= 8:
            return True
    return False



def _extract_desc_unit_price_hint(desc):
    s = str(desc or '')
    if not s:
        return None
    cands = []
    for m in re.finditer(r'(?i)(?:[$€]\s*)?(\d{1,2}[.,]\d{1,4})(?:\s*[$€]|(?:\s*(?:eur|usd)))?', s):
        tok = m.group(1)
        post = s[m.end():m.end()+8].lower()
        # Ignore volume/alcohol tokens like "0,75 l", "13,0%".
        if re.search(r'^\s*(l|ml|cl|kg|g)\b', post) or post.strip().startswith('%'):
            continue
        try:
            v = float(tok.replace(',', '.'))
        except Exception:
            continue
        if 0.2 <= v <= 40.0:
            cands.append(v)
    if not cands:
        return None
    return min(cands)

def _validate_line_items(items):
    fixed = []
    for raw_item in items:
        item = dict(raw_item)
        item['description'] = _clean_line_item_description(item.get('description'), item)

        qty  = item.get('quantity')
        up   = item.get('unit_price')
        la   = item.get('line_amount')
        desc = str(item.get('description',''))[:60]

        if qty is None or up is None or la is None or qty == 0:
            if not _is_noise_line_item(item):
                fixed.append(item)
            continue

        qty, up, la = float(qty), float(up), float(la)
        computed = round(up * qty, 4)
        tol = max(abs(la) * 0.02, 0.05)
        if abs(computed - la) <= tol:
            # Arithmetic can still be wrong by a hidden pack scale (e.g., 28 x 61.2 instead of 336 x 5.1).
            hint = _extract_desc_unit_price_hint(item.get('description'))
            pack_mult = _extract_pack_multiplier(item.get('description'))
            scaled = False
            if hint and hint > 0:
                ratio = up / hint if hint else None
                if ratio and 7.5 <= ratio <= 30.0:
                    cand_qty = la / hint
                    near_int = abs(cand_qty - round(cand_qty)) <= max(0.3, cand_qty * 0.01)
                    pack_ok = (pack_mult is None) or (abs(ratio - pack_mult) <= max(2.0, pack_mult * 0.35))
                    if near_int and pack_ok and cand_qty > qty * 3:
                        item['unit_price'] = round(hint, 6)
                        item['quantity'] = round(cand_qty, 4)
                        print(f"    Fix(S desc-scale {ratio:.1f}x): '{desc}' up={up}->{hint:.4f}, qty={qty}->{cand_qty:.2f}")
                        scaled = True
            if not scaled and pack_mult and pack_mult >= 8 and up > 20 and qty < 200:
                cand_up = up / pack_mult
                cand_qty = qty * pack_mult
                if 0.2 <= cand_up <= 30 and abs((cand_up * cand_qty) - la) <= tol:
                    item['unit_price'] = round(cand_up, 6)
                    item['quantity'] = round(cand_qty, 4)
                    print(f"    Fix(S pack-scale {pack_mult:.0f}x): '{desc}' up={up}->{cand_up:.4f}, qty={qty}->{cand_qty:.2f}")
            if not _is_noise_line_item(item):
                fixed.append(item)
            continue

        if la != 0 and abs(up - la) / max(abs(la), 1) < 0.05:
            item['unit_price'] = round(la / qty, 6)
            print(f"    Fix(A): '{desc}' up={up}->{item['unit_price']:.4f}")
            if not _is_noise_line_item(item):
                fixed.append(item)
            continue

        found = False
        if up != 0 and la != 0:
            correct_qty = la / up
            ratio = correct_qty / max(abs(qty), 1e-9)
            pack_mult = _extract_pack_multiplier(item.get('description'))
            near_int = abs(correct_qty - round(correct_qty)) <= max(abs(correct_qty) * 0.01, 0.2)
            qty_looks_underexpanded = (
                (pack_mult and abs(ratio - pack_mult) <= max(pack_mult * 0.25, 0.5)) or
                (qty <= 5 and correct_qty >= 20) or
                (ratio >= 8 and near_int)
            )
            qty_looks_overexpanded = (
                (ratio <= 0.2 and near_int and qty >= 200) or
                (pack_mult and ratio <= 0.2 and near_int)
            )
            if (qty_looks_underexpanded or qty_looks_overexpanded) and abs(correct_qty - qty) / max(abs(qty), 1) > 0.05:
                item['quantity'] = round(correct_qty, 4)
                print(f"    Fix(B {ratio:.1f}x): '{desc}' qty={qty}->{correct_qty:.2f}")
                found = True

        if not found and qty != 0 and la != 0:
            correct_up = la / qty
            ratio = correct_up / max(abs(up), 1e-9)
            if correct_up > 0 and abs(correct_up - up) / max(abs(up), 1) > 0.05:
                if 0.1 <= ratio <= 3.0:
                    item['unit_price'] = round(correct_up, 6)
                    print(f"    Fix(C {ratio:.1f}x): '{desc}' up={up}->{correct_up:.4f}")
                    found = True

        if not found and la not in (None, 0.0):
            desc_txt = str(item.get('description') or '')
            for m in re.finditer(r'(?<!\d)(\d{1,3}[.,]\d{1,4})(?:\s*(?:[$]|EUR|USD|\u20ac))?(?!\d)', desc_txt):
                tok = m.group(1)
                chunk = m.group(0)
                # Skip non-price decimals such as alcohol percentages and volumes (e.g., 0,75 L).
                post = desc_txt[m.end():m.end()+8]
                if '%' in post:
                    continue
                if re.search(r'^\s*(?:l|ml|cl|kg|g)\b', post.lower()):
                    continue
                try:
                    cand_up = float(tok.replace(',', '.'))
                except Exception:
                    continue
                if cand_up <= 0 or cand_up > 200:
                    continue
                has_currency = any(sym in chunk.upper() for sym in ('$', 'EUR', 'USD', '\u20ac'))
                # Without explicit currency, keep a tighter plausible unit-price range.
                if (not has_currency) and not (0.2 <= cand_up <= 20.0):
                    continue
                if abs(cand_up - up) / max(abs(up), 1.0) < 0.10:
                    continue

                cand_qty = la / cand_up
                if cand_qty <= 0:
                    continue
                near_int_qty = abs(cand_qty - round(cand_qty)) <= max(0.25, cand_qty * 0.01)
                if near_int_qty and abs((cand_qty * cand_up) - la) <= max(abs(la) * 0.02, 0.05):
                    item['unit_price'] = round(cand_up, 6)
                    item['quantity'] = round(cand_qty, 4)
                    print(f"    Fix(E desc price): '{desc}' up={up}->{cand_up:.4f}, qty={qty}->{cand_qty:.2f}")
                    found = True
                    break

        if not found:
            print(f"    No safe fix: '{desc}' {up}x{qty}={computed:.2f} la={la}")

        if not _is_noise_line_item(item):
            fixed.append(item)
    return fixed


Line item validator ready.


## 15  Header text extraction

In [16]:
def pdfminer_to_text(pdf_path):
    pages_md = []
    try:
        with open(pdf_path, 'rb') as f:
            for pn, page_layout in enumerate(extract_pages(f), 1):
                boxes = []
                for el in page_layout:
                    if isinstance(el, LTTextBox):
                        t = el.get_text().strip()
                        if t:
                            boxes.append({'text': t, 'y1': el.y1, 'x0': el.x0})
                if not boxes:
                    continue
                boxes_sorted = sorted(boxes, key=lambda b: -b['y1'])
                rows, current = [], [boxes_sorted[0]]
                for box in boxes_sorted[1:]:
                    if abs(box['y1'] - current[-1]['y1']) < 8:
                        current.append(box)
                    else:
                        rows.append(sorted(current, key=lambda b: b['x0']))
                        current = [box]
                if current:
                    rows.append(sorted(current, key=lambda b: b['x0']))
                page_lines = [f'## Page {pn}']
                for row in rows:
                    if len(row) == 1:
                        page_lines.append(row[0]['text'])
                    else:
                        page_lines.append('| ' + ' | '.join(r['text'].replace('\n', ' ') for r in row) + ' |')
                pages_md.append('\n'.join(page_lines))
    except Exception as e:
        print(f'    pdfminer error: {e}')
        return ''
    return normalise_numbers('\n\n'.join(pages_md))


def scanned_pages_to_text(pdf_path):
    doc = pymupdf.open(pdf_path)
    n_pages = len(doc)
    doc.close()
    pages = []
    for pn in range(n_pages):
        arr = _rasterise_page(pdf_path, pn)
        raw = pytesseract.image_to_string(Image.fromarray(arr), config='--oem 1 --psm 6')
        pages.append(f'## Page {pn+1}\n{raw}')
    return normalise_numbers('\n\n'.join(pages))


def _page_img_fraction(page):
    try:
        page_area = page.rect.width * page.rect.height
        if page_area <= 0:
            return 0.0
        image_area = 0.0
        for b in page.get_text('dict', flags=0).get('blocks', []):
            if b.get('type') == 1:
                bb = b['bbox']
                image_area += max(0, bb[2]-bb[0]) * max(0, bb[3]-bb[1])
        if image_area < 1.0 and page.get_images(full=False):
            image_area = page_area * 0.90
        return min(max(image_area / page_area, 0.0), 1.0)
    except Exception:
        return 0.0




def _text_layer_is_reliable(txt):
    t = str(txt or '').strip()
    if len(t) < 180:
        return False
    alnum = sum(ch.isalnum() for ch in t)
    ratio = alnum / max(len(t), 1)
    if ratio < 0.55:
        return False
    weird = len(re.findall(r"[~^`_|]{3,}|[=]{4,}|[><]{4,}", t))
    if weird >= 2:
        return False
    return True

def hybrid_pages_to_text(pdf_path, min_text_chars=220):
    """
    Mixed-mode extraction:
      - per page, use text layer when reliable
      - fallback to OCR when page is raster-dominant or text is sparse
    """
    doc = pymupdf.open(pdf_path)
    pages = []
    for pn in range(len(doc)):
        page = doc[pn]
        txt = (page.get_text('text') or '').strip()
        img_frac = _page_img_fraction(page)
        txt_ok = _text_layer_is_reliable(txt)
        use_ocr = (not txt_ok) or (len(txt) < min_text_chars) or (img_frac > 0.75 and len(txt) < 1200)
        if use_ocr:
            arr = _rasterise_page(pdf_path, pn)
            raw = pytesseract.image_to_string(Image.fromarray(arr), config='--oem 1 --psm 6')
            pages.append(f'## Page {pn+1}\n{raw}')
        else:
            pages.append(f'## Page {pn+1}\n{txt[:5000]}')
    doc.close()
    return normalise_numbers('\n\n'.join(pages))


print('Header text extraction ready.')




def _split_docling_pages(md: str):
    txt = str(md or '')
    if not txt.strip():
        return []
    if re.search(r'(?m)^##\s+Page\s+\d+', txt):
        pages = re.split(r'(?m)(?=^##\s+Page\s+\d+)', txt)
        return [p.strip() for p in pages if p.strip()]
    if '<!-- image -->' in txt:
        parts = [p.strip() for p in txt.split('<!-- image -->') if p.strip()]
        return parts
    if '\f' in txt:
        parts = [p.strip() for p in txt.split('\f') if p.strip()]
        return parts
    return [txt.strip()]


def _invoice_markdown_quality(md: str):
    t = str(md or '')
    if not t.strip():
        return 0.0
    chars = len(t)
    digit_tokens = len(re.findall(r'(?<!\d)\d{1,4}(?:[.,]\d{1,4})?(?!\d)', t))
    money_tokens = len(re.findall(r'(?i)(?:\$|€|eur|usd|total|subtotal|amount\s+due|invoice)', t))

    page_bonus = 0.0
    try:
        parts = _split_docling_pages(t)
        if parts:
            scored = 0
            for p in parts:
                ps = _score_page(p)
                if ps.get('invoice', 0) >= 1:
                    scored += 1
            page_bonus = min(scored, 3) * 0.8
    except Exception:
        pass

    score = 0.0
    score += min(chars / 2500.0, 3.0)
    score += min(digit_tokens / 18.0, 3.0)
    score += min(money_tokens / 8.0, 2.0)
    score += page_bonus
    return score


def docling_pages_to_text(pdf_path):
    if DocumentConverter is None:
        return ''
    # Some environments force a dead local proxy (127.0.0.1:9), which blocks Docling model fetches.
    proxy_keys = ['HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'http_proxy', 'https_proxy', 'all_proxy']
    saved_env = {k: os.environ.get(k) for k in proxy_keys}
    try:
        for k in proxy_keys:
            os.environ.pop(k, None)

        conv = DocumentConverter()
        res = conv.convert(pdf_path)
        doc = getattr(res, 'document', None)
        if doc is None:
            return ''
        md = ''
        try:
            md = doc.export_to_markdown()
        except Exception:
            md = str(doc)
        pages = _split_docling_pages(md)
        if not pages:
            return ''
        out = []
        for i, p in enumerate(pages, 1):
            block = p
            if not re.search(r'(?m)^##\s+Page\s+\d+', p):
                block = f'## Page {i}\n{p}'
            out.append(block)
        return normalise_numbers('\n\n'.join(out))
    except Exception as e:
        print(f'    docling error: {e}')
        return ''
    finally:
        for k, v in saved_env.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v


Header text extraction ready.


## 16  Page classifier

In [17]:
_INVOICE_SIGNALS = [
    "invoice n\u00b0", "invoice no.", "invoice #", "invoice number",
    "facture n\u00b0", "n\u00b0 facture", "facture",
    "factura", "n\u00famero de factura", "rechnungsnummer", "fattura",
    "total to pay", "balance due", "amount due", "total amount due",
    "net a payer", "montant ht", "montant net", "total ttc",
    "total a pagar", "gesamtbetrag",
    "iban", "bank details", "remit to", "wire transfer",
    "swift", "bic", "sort code", "compte bancaire",
    "incoterm", "payment terms", "net 30", "net 60",
    "vat number", "tva", "tax invoice", "invoice", "commercial invoice", "proforma",
]
_PO_SIGNALS = [
    "purchase order", "order confirmation", "confirmation of order",
    "p.o. number", "order number", "order no.",
    "authorized signature", "approved by", "buyer id",
    "requested by", "requisition",
    "ext. price", "extended price", "disc. %",
    "trade discount", "order total", "order qty",
    "packing list", "bill of lading", "b/l number", "b/l no.",
    "airway bill", "awb number",
    "delivery note", "dispatch note", "consignment note",
    "ship 30day", "ship 45day", "ship 60day", "vendor id", "vend-tr",
]
_PAGE_SEP = re.compile(r'(?=<!-- image -->|^## (?:Page )?\d)', re.MULTILINE)


def _score_page(text):
    lo = text.lower()
    return {'invoice': sum(1 for s in _INVOICE_SIGNALS if s in lo),
            'po':      sum(1 for s in _PO_SIGNALS      if s in lo)}




def identify_invoice_pages(markdown):
    raw_pages = _PAGE_SEP.split(markdown)
    if len(raw_pages) <= 1:
        return markdown, 'Single-page document.', [1]
    invoice_pages, po_pages = [], []
    for i, page in enumerate(raw_pages, 1):
        if not page.strip(): continue
        scores = _score_page(page)
        if scores['po'] > scores['invoice'] and scores['po'] >= 3:
            po_pages.append(i)
        elif scores['invoice'] >= 1:
            invoice_pages.append((i, page))
    if invoice_pages:
        filtered_md = '\n\n'.join(p for _, p in invoice_pages)
        note = (
            f"Document has {len(raw_pages)} pages. "
            f"Vendor invoice pages (USE THESE): {[i for i, _ in invoice_pages]}. "
            f"Excluded pages (purchase orders / logistics docs): {po_pages}. "
            f"Extract ALL fields ONLY from the vendor invoice pages."
        )
        return filtered_md, note, [i for i, _ in invoice_pages]
    return markdown, f'Document has {len(raw_pages)} pages. All pages retained.', [1]

def _select_pages_by_index(markdown, page_indices):
    pages = [p for p in _PAGE_SEP.split(str(markdown or '')) if p.strip()]
    idx = sorted({int(i) for i in (page_indices or []) if int(i) >= 1})
    if not pages or not idx:
        return str(markdown or '')
    picked = []
    for i in idx:
        if 1 <= i <= len(pages):
            picked.append(pages[i - 1])
    return '\n\n'.join(picked) if picked else str(markdown or '')


Page classifier ready.


## 17  Letterhead anchor (fixes: buyer ID pattern, min length, bad OCR)

**Three new fixes on top of v7.2:**
1. **`buyer` keyword added to `_BAD_ANCHOR_RE`** — catches `'Buyer!ID dbrown'`,
   `'Buyer ID = jmartinez'`, `'Buyer !ID_ ksewell'` which were all slipping through
   and becoming the letterhead fallback, producing wrong vendor names.
2. **`_is_bad_anchor()` strict check** — validates the accepted line isn't a
   document-type label, address fragment, or buyer reference before returning it.
3. **Page scan breadth increased** — scans top 50% of page (was 40%) to capture
   company names that sit below a logo block.

In [18]:
# Patterns that definitively rule out a line as a vendor company name.
# Key addition: 'buyer' catches 'Buyer ID', 'Buyer!ID', 'Buyer !ID_' etc.
_BAD_ANCHOR_RE = re.compile(
    r'(?i)'
    r'(^\s*\d)'
    r'|(facture|douane|invoice|proforma|commercial|bordereau|avis)'
    r'|(\brue\b|\broute\b|\bbp\b|\bcedex\b|\bavenue\b|\bstreet\b)'
    r'|(au capital|capital de|siren|siret|\brcs\b|euros|chiffre)'
    r'|(join\w+|reglement|changed\s+since\s+the\s+previous\s+revision)'
    r'|(\bbuyer\b|\bbill\s+to\b|\bship\s+to\b|\bsold\s+to\b)'
    r'|(\bid\b.*\b[a-z]{3,10}\b$)'
    r'|(\(\d{3}\)\s*[\d\s\-\.]{6,})'
    r'|(\bext\.?\s*\d{3,}\b)'
    r'|(,\s*$)'
)

_COMPANY_LINE_RE = re.compile(
    r'^(?!.*(?:bill to|ship to|sold to|deliver to|attention|attn|page \d))'
    r'(?=.*[A-Za-z]{3,})'
    r'(?=.*(?:[A-Z]{2,}|(?:[A-Za-z]+\s){1,}))'
    r'.{4,80}$', re.IGNORECASE
)


def _is_buyer_text(s):
    """Return True if the string identifies the buyer (CPJ / Caribbean side)."""
    lo = s.lower()
    return any(b in lo for b in _BUYER_STRINGS)


def _top_lines_pdfminer(pdf_path, page_num=0, n_lines=20):
    lines = []
    try:
        with open(pdf_path, 'rb') as f:
            for i, page_layout in enumerate(extract_pages(f)):
                if i != page_num: continue
                boxes = sorted([e for e in page_layout if isinstance(e, LTTextBox)],
                               key=lambda e: -e.y1)
                for box in boxes:
                    for lt in box.get_text().splitlines():
                        t = lt.strip()
                        if t: lines.append(t)
                        if len(lines) >= n_lines: break
                    if len(lines) >= n_lines: break
                break
    except Exception:
        pass
    return lines


def _top_lines_tesseract(pdf_path, page_num=0, n_lines=30):
    arr = _rasterise_page(pdf_path, page_num)
    h = arr.shape[0]
    top_strip = arr[:int(h * 0.50), :]    # top 50% (was 40%)
    img_pil = Image.fromarray(top_strip)
    raw = pytesseract.image_to_string(img_pil, config='--oem 1 --psm 6')
    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    return lines[:n_lines]


def _looks_like_bad_vendor_hint(s):
    s = str(s or '').strip()
    if not s:
        return True
    if _BAD_ANCHOR_RE.search(s):
        return True
    if re.search(r'(?i)\b\d{1,2}/\d{1,2}/\d{2,4}\b', s) and re.search(r'[?$]', s):
        return True
    if len(re.findall(r'\d', s)) >= 6 and len(re.findall(r'[A-Za-z]{3,}', s)) < 2:
        return True
    return False



def extract_letterhead_anchor(pdf_path, scanned, page_num=0):
    """
    Return the vendor company name from the top of the invoice page.

    Filters applied in order:
      1. Skip lines shorter than 8 chars (noise)
      2. Skip lines that match _BAD_ANCHOR_RE (includes buyer/address keywords)
      3. Skip lines that contain buyer identity strings (CPJ/Caribbean etc.)
      4. Prefer lines matching _COMPANY_LINE_RE (looks like a company name)
      5. Last resort: first non-bad, non-buyer line >= 8 chars
    """
    lines = (_top_lines_tesseract(pdf_path, page_num)
             if scanned else _top_lines_pdfminer(pdf_path, page_num))
    if not lines:
        return ''

    for line in lines:
        s = line.strip()
        if len(s) < 8:
            continue
        if _looks_like_bad_vendor_hint(s):
            continue
        if _is_buyer_text(s):
            continue
        if _COMPANY_LINE_RE.match(s):
            return s

    # Last resort
    for line in lines:
        s = line.strip()
        if len(s) >= 8 and not _looks_like_bad_vendor_hint(s) and not _is_buyer_text(s):
            return s
    return ''


print('Letterhead anchor ready.')


Letterhead anchor ready.


## 18  Load LLM (Llama-3.1-8B 4-bit NF4)

In [19]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} in 4-bit NF4...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)

slm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    return_full_text=False,
)

print("Model loaded.")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"  GPU {i}: {alloc:.1f} GB / {total:.0f} GB")


Loading meta-llama/Llama-3.1-8B-Instruct in 4-bit NF4...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded.
  GPU 0: 1.6 GB / 15 GB
  GPU 1: 3.8 GB / 15 GB


## 19  LLM prompts — header fields only (v8 improvements)

**Header quality fixes:**
- `invoice_number`: now strips bare `INVOICE ` prefix (no N°/# suffix needed)
- `invoice_date`: explicit EU/US ambiguity rule — non-US vendor → day-first
- `sub_total`: hard constraint — sub_total < total when freight/insurance > 0
- `ship_to` / `bill_to`: prompt now demands the FULL address block, not just company name
- `vendor_address`: explicitly ask for the address in the header/letterhead area

In [20]:
ROLE_SYSTEM = (
    "You are a document analysis assistant. "
    "Identify the trading parties in an invoice and return JSON only."
)

_ROLE_USER_TEMPLATE = (
    "LETTERHEAD HINT (text from the top of the invoice — likely the vendor):\n"
    "  {letterhead_hint}\n\n"
    "VENDOR = the company that ISSUED this invoice (the seller).\n"
    "  - Name is at the top of the page in the letterhead.\n"
    "  - If LETTERHEAD HINT looks like a company name, use it verbatim.\n"
    "  - 'Caribbean Producers' / 'CPJ' is ALWAYS the BUYER, never the vendor.\n"
    "  - 'Buyer ID', 'Buyer !ID_', buyer usernames (ksewell, dbrown…) are NOT vendor names.\n\n"
    "BUYER = company that RECEIVED the invoice.\n"
    "  - Appears under Bill To / Ship To / Sold To / Customer.\n\n"
    "Return ONLY this JSON (no fences, no prose):\n"
    '{{"vendor_name": "...", "vendor_address": "full address or null", '
    '"buyer_name": "...", "buyer_address": "full address or null"}}\n\n'
    "--- INVOICE MARKDOWN ---\n{markdown}\n--- END ---"
)


EXTRACTION_SYSTEM_PROMPT = (
    "You are a structured data extraction engine.\n"
    "Extract ALL fields — header AND line items — from the invoice Markdown below.\n\n"
    "OUTPUT RULES:\n"
    "- Single JSON object. No prose, no markdown fences.\n"
    "- Use null for absent fields. Numbers: plain floats. Dates: YYYY-MM-DD.\n"
    "- Numbers are pre-normalised (EU commas already converted to dots).\n\n"
    "═══ HEADER FIELD RULES ═══\n\n"
    "vendor_name:\n"
    "  Use EXACTLY the vendor from PAGE CONTEXT. CPJ / Caribbean Producers = BUYER, never vendor.\n\n"
    "vendor_address:\n"
    "  Full mailing address of the vendor (street + city + postal + country).\n"
    "  Appears in the invoice header/letterhead, usually below the vendor name.\n"
    "  Do NOT use the buyer's address. Do NOT include phone/fax in this field.\n\n"
    "invoice_number:\n"
    "  Bare code only — e.g. '93184726', '160196', '2023/5275'.\n"
    "  Strip ALL prefixes: 'Invoice N°', 'Facture N°', 'INVOICE ', '#', 'N°'.\n"
    "  REJECT: BL numbers (start with BL), delivery notes, PO numbers, customer refs.\n"
    "  If multiple invoice-like numbers appear, use only the one labelled Invoice/Facture.\n\n"
    "purchase_order_number:\n"
    "  Full PO code including prefix: 'POH112330', 'PO H110077', 'PC#H110051'.\n"
    "  If ONLY one PO-style code appears, use it. Do NOT concatenate multiple codes.\n\n"
    "invoice_date:\n"
    "  Date vendor issued invoice. Output YYYY-MM-DD.\n"
    "  European vendors (France/Netherlands/Chile): DD/MM/YYYY → YYYY-MM-DD.\n"
    "  US vendors (MA/FL/TX state in address): MM/DD/YYYY → YYYY-MM-DD.\n"
    "  Labels: Invoice Date, Date, Date de facture. NOT due date, NOT BL date.\n\n"
    "sub_total:\n"
    "  Merchandise total BEFORE freight, insurance, taxes.\n"
    "  = sum of all line_item line_amounts.\n"
    "  Labels: Subtotal, Net Amount, Montant HT, Total HT.\n"
    "  CRITICAL: sub_total < total when freight or insurance is non-zero.\n"
    "  'Total to pay', 'Total', 'Grand Total' → use for 'total' field, NOT sub_total.\n\n"
    "shipping_handling_charge:\n"
    "  ONLY for: Sea Freight, Air Freight, Ocean Freight, Fret, Handling. Use 0.0 when zero.\n\n"
    "insurance_charge:\n"
    "  ONLY for: Insurance, Assurance. Use 0.0 when zero.\n\n"
    "vendor_phone:\n"
    "  Only extract if an explicit phone/tel label precedes digits. Null if absent.\n\n"
    "ship_to / bill_to:\n"
    "  COMPLETE address block: company name + street + city + country. Do not truncate.\n\n"
    "═══ LINE ITEM RULES (CRITICAL) ═══\n\n"
    "Extract every PRODUCT ROW as a line item.\n\n"
    "SKIP these — they are SCALAR fields, NOT line items:\n"
    "  • Sea/Ocean/Air Freight, Handling  → shipping_handling_charge\n"
    "  • Insurance, Assurance             → insurance_charge\n"
    "  • Subtotal, Total, VAT, Tax rows\n"
    "  • Terms, conditions, compliance text\n\n"
    "description:\n"
    "  Keep the FULL product description needed to distinguish the SKU.\n"
    "  Preserve pack/size/variant text such as 4x2500g, 5x5lbs, 12/750ML, reg3, cut/style, vintage, and similar qualifiers.\n"
    "  Strip only leading row counters, leading case-count prefixes, isolated material/item codes before the name, and trailing monetary columns that leaked into the text.\n"
    "  Do NOT shorten descriptions to a generic family name if packaging or variant text is present on the row.\n"
    "  Example: '14 84 BOUT. DE LADOUCETTE POUILLY FUME 2022 13,5? 750 MLB 14,40 1209,60'\n"
    "    ? description='DE LADOUCETTE POUILLY FUME 2022 13,5? 750 MLB', quantity=84, unit_price=14.40, line_amount=1209.60\n\n"
    "quantity:\n"
    "  Count of INDIVIDUAL UNITS (bottles, pieces, kg) — NOT cases.\n"
    "  If the row shows CASES and description contains a pack size:\n"
    "    cases × pack_size = quantity\n"
    "    '28 CS' + '12/750ML' → quantity=336  (28 × 12)\n"
    "    '70 cases' + 'Pack: 10 × 1 Lb' → quantity=700  (70 × 10)\n"
    "    '84 BOUT.' in description means 84 bottles → quantity=84\n\n"
    "unit_price:\n"
    "  Price per INDIVIDUAL UNIT. Must satisfy: unit_price × quantity ≈ line_amount.\n"
    "  If the table shows case price, divide by pack size.\n\n"
    "line_amount:\n"
    "  Row total as printed. Verify: quantity × unit_price ≈ line_amount (within 5%).\n"
    "  If they don't match, recheck which column is which.\n"
)


def build_role_prompt(markdown, letterhead_hint=''):
    truncated = markdown[:MAX_MARKDOWN_CHARS // 2]
    hint = letterhead_hint or '(not available)'
    return _ROLE_USER_TEMPLATE.format(letterhead_hint=hint, markdown=truncated)


def build_extraction_prompt(markdown, schema, page_note=''):
    if len(markdown) > MAX_MARKDOWN_CHARS:
        print(f'    Truncated: {len(markdown):,} -> {MAX_MARKDOWN_CHARS:,} chars')
        markdown = markdown[:MAX_MARKDOWN_CHARS]
    note = f'\nPAGE CONTEXT:\n{page_note}\n' if page_note else ''
    return (
        'Extract ALL invoice fields (header + line items) from the Markdown below.\n'
        'Return ONLY a valid JSON object matching this schema:\n\n'
        f'{schema}\n{note}\n'
        '--- INVOICE MARKDOWN ---\n'
        f'{markdown}\n--- END ---'
    )


# Keep backward-compat alias
build_header_prompt = build_extraction_prompt
print('Prompts ready.')


LINE_ITEM_SYSTEM_PROMPT = (
    "You extract PRODUCT line items from invoice table text only.\n"
    "Return JSON only. No prose, no markdown fences.\n"
    "Read the text as a noisy OCR dump of the product table.\n"
    "Use ONLY product rows. Skip totals, freight, insurance, tax, and headers.\n"
    "Preserve distinguishing pack and variant text like 4x2500g, 5x5lbs, 12/750ML, reg3, cut/style, vintage, and similar qualifiers.\n"
    "Do not append trailing quantity/unit columns to description.\n"
    "Do not keep isolated language markers like FR or ES at the end of descriptions.\n"
    "Quantity must be the individual-unit count when the row clearly implies cases ? pack size.\n"
    "Unit price must be the per-unit price. Verify quantity ? unit_price ? line_amount.\n"
)


def build_line_item_prompt(table_text, page_note=''):
    text = str(table_text or '').strip()
    if len(text) > MAX_MARKDOWN_CHARS:
        text = text[:MAX_MARKDOWN_CHARS]
    note = f'\nPAGE CONTEXT:\n{page_note}\n' if page_note else ''
    return (
        'Extract ONLY product line items from the table OCR text below.\n'
        'Return ONLY a valid JSON object matching this schema:\n\n'
        '{"line_items": ' + LINE_ITEM_SCHEMA_STR + '}\n'
        f'{note}\n'
        '--- TABLE OCR TEXT ---\n'
        f'{text}\n--- END ---'
    )


Prompts ready.


## 20  SLM helpers and post-processing

In [21]:
def call_slm(system, user, max_new_tokens=None):
    messages = [{"role": "system", "content": system},
                {"role": "user",   "content": user}]
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    kwargs = {
        'eos_token_id': [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],
    }
    if max_new_tokens is not None:
        kwargs['max_new_tokens'] = max_new_tokens
    outputs = slm_pipeline(messages, **kwargs)
    return outputs[0]["generated_text"]


def _strip_fences(text):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        end = -1 if lines[-1].strip() == "```" else len(lines)
        text = "\n".join(lines[1:end])
    return text.strip()


_INV_NUM_RE = re.compile(
    r"(?i)^\s*(?:"
    r"(?:tax\s+)?invoice\s*(?:n[o°.#]?\s*|number\s*|no\.?\s*)?"
    r"|facture\s*(?:n[o°.]?\s*)?"
    r"|n\s*[°o.]\s*"           # catches "N°", "No.", "N°", "N ° "
    r"|n\s+[°]\s*"             # "N °" with space before degree
    r"|#\s*"
    r")"
)
_BL_RE = re.compile(r"(?i)^bl", re.IGNORECASE)
_PO_TOKEN_RE = re.compile(r'(?i)\b(?:PO[HJ]?\s*#?\s*[A-Z0-9]+|PC#\s*[A-Z0-9]+|H\d{5,})\b')
_FREIGHT_TEXT_RE = re.compile(r'(?i)\b(ocean\s+freight|air\s+freight|sea\s+freight|freight(?:\s+charge)?|handling\s+charge|shipping\s+charge|insurance|assurance|surcharge)\b')


def _clean_invoice_number(raw):
    if raw is None:
        return None
    cleaned = _INV_NUM_RE.sub("", str(raw).strip()).strip()
    if not cleaned:
        return None
    if _BL_RE.match(cleaned):
        print(f"    invoice_number: rejected BL '{cleaned}'")
        return None
    return cleaned


def _clean_po_number(raw):
    if raw is None:
        return None
    s = str(raw).strip().upper()
    m = _PO_TOKEN_RE.search(s)
    if not m:
        return s or None
    token = re.sub(r'\s+', ' ', m.group(0)).strip()
    token = token.replace('PO #', 'PO#').replace('PC #', 'PC#')
    if token.startswith('POH') or token.startswith('POJ') or token.startswith('PC#'):
        return token.replace(' ', '')
    return token


def _normalize_date(raw, vendor_address=None):
    if raw is None:
        return None
    s = str(raw).strip()
    if re.match(r"^\d{4}-\d{2}-\d{2}$", s):
        return s
    try:
        from dateutil import parser as dparser
        us_states = {"MA", "FL", "TX", "CA", "NY", "NJ", "GA", "IL", "OH", "PA"}
        is_us = False
        if vendor_address:
            addr_upper = str(vendor_address).upper()
            is_us = any(f" {st}" in addr_upper or f",{st}" in addr_upper for st in us_states)
        return dparser.parse(s, dayfirst=not is_us).strftime("%Y-%m-%d")
    except Exception:
        return s


def _validate_phone(raw):
    if raw is None:
        return None
    s = str(raw).strip()
    if len(re.findall(r"\d", s)) < 7:
        return None
    if re.search(r'(?i)ext\.?\s*0{3,}', s):
        return None
    s = re.sub(r'\s*-\s*', '-', s)
    s = re.sub(r'\s{2,}', ' ', s).strip()
    return s





def _repair_vendor_name(name, markdown):
    cur = str(name or '').strip()
    if not cur:
        return cur
    if not re.match(r'^(?:[A-Z]\.\s*){4,}', cur):
        return cur

    lines = _markdown_lines(markdown)
    best = cur
    best_score = -1.0
    for ln in lines:
        lo = ln.lower()
        if any(k in lo for k in ['ship to', 'bill to', 'buyer id', 'invoice', 'total']):
            continue
        if re.search(r'(?i)\b(r\.?c\.?s|site|www\.|payment|virement|iban|swift|code|recipient)\b', ln):
            continue
        ln = re.split(r'(?i)\bcaribbean\s+producers\b', ln)[0].strip(' ,;:-')
        if any(ch.isdigit() for ch in ln):
            continue
        words = re.findall(r"[A-Za-z][A-Za-z'.-]+", ln)
        if len(words) < 2 or len(words) > 6:
            continue
        upperish = sum(1 for w in words if w[:1].isupper())
        if upperish < max(2, len(words) - 1):
            continue
        cand = ' '.join(words[:6]).strip()
        if len(cand) < 8:
            continue
        score = len(cand) + upperish * 3
        if score > best_score:
            best = cand
            best_score = score
    return best

def _clean_vendor_address(raw):
    if raw is None:
        return None
    s = str(raw).strip()
    s = re.sub(r'\s+(?:Phone|Fax|Tel|T?l|Telephone|T?l?phone)\s*:.*$', '', s, flags=re.IGNORECASE).strip()
    s = re.sub(r'\s+\(\d{3}\)\s*\d[\d\s\-\.]{6,}\s*$', '', s).strip()
    s = re.sub(r'\s*,\s*', ', ', s)
    s = re.sub(r'\s{2,}', ' ', s).strip()

    parts = [t.strip(' ,;/') for t in re.split(r'\s*/\s*', s) if t.strip()]
    po_parts = [t for t in parts if re.search(r'(?i)\b(?:P\.?\s*O\.?\s*Box|Postbus|BP\b|Bo[i?]te\s+postale)\b', t)]
    if po_parts:
        s = max(po_parts, key=len)

    return s or None


def _infer_vendor_name_from_markdown(markdown):
    lines = _markdown_lines(markdown)
    if not lines:
        return None
    street_re = re.compile(r'(?i)\b(route|road|rd\.?|street|st\.?|avenue|ave\.?|way|boulevard|blvd\.?|drive|dr\.?|lane|ln\.?|box|unit|suite|ste\.?|floor)\b')
    stop_re = re.compile(r'(?i)\b(ship\s*to|bill\s*to|buyer|customer|invoice|total|subtotal|tax|date|po\b|purchase order|delivery note|iban|swift|phone|fax|email)\b')

    best = None
    best_score = -1.0
    for ln in lines[:30]:
        if _is_buyer_text(ln) or stop_re.search(ln):
            continue
        if any(ch.isdigit() for ch in ln) or street_re.search(ln):
            continue
        words = re.findall(r"[A-Za-z][A-Za-z'.&-]+", ln)
        if len(words) < 2 or len(words) > 8:
            continue
        upperish = sum(1 for w in words if w[:1].isupper())
        if upperish < max(2, len(words) - 2):
            continue
        cand = ' '.join(words[:8]).strip(' ,;:-')
        score = len(cand) + upperish * 3
        if score > best_score:
            best = cand
            best_score = score
    return best


def _infer_vendor_address_from_markdown(markdown, vendor_name=''):
    lines = _markdown_lines(markdown)
    if not lines:
        return None
    name_tokens = set(re.findall(r'[a-z]+', str(vendor_name or '').lower()))
    stop_re = re.compile(r'(?i)\b(ship\s*to|bill\s*to|buyer|customer|invoice|total|subtotal|tax|delivery note|iban|swift|phone|fax|email)\b')
    street_re = re.compile(r'(?i)\b(route|road|rd\.?|street|st\.?|avenue|ave\.?|way|boulevard|blvd\.?|drive|dr\.?|lane|ln\.?|box|unit|suite|ste\.?|floor|gassin|pauillac|france|chile|holland|steenderen|coral|gables|ma|fl)\b')

    for i, ln in enumerate(lines[:40]):
        if stop_re.search(ln) or _is_buyer_text(ln):
            continue
        words = set(re.findall(r'[a-z]+', ln.lower()))
        if name_tokens and len(name_tokens & words) < max(1, min(2, len(name_tokens))):
            continue
        window = ' '.join(lines[i+1:i+3]).strip()
        if not window:
            continue
        if _is_buyer_text(window) or stop_re.search(window):
            continue
        if street_re.search(window) or re.search(r'\b\d{4,6}\b', window):
            return _clean_vendor_address(window)
    return None

def _looks_like_party_line(line):
    s = str(line or '').strip()
    if not s or len(s) > 140:
        return False
    if re.search(r'(?i)\b(www\.|https?://|email|fax|phone|tel\.?|ext\.?)\b', s):
        return False
    if _PARTY_STOP_RE.search(s) and not _PARTY_LABEL_RE.search(s):
        return False

    has_company = bool(_PARTY_COMPANY_RE.search(s))
    has_addr = bool(_PARTY_ADDR_HINT_RE.search(s) or re.search(r'\b\d{1,5}\b', s))
    upper_words = re.findall(r'\b[A-Z][A-Z.&-]{2,}\b', s)
    return has_company or has_addr or len(upper_words) >= 2


def _trim_mixed_party_prefix(text):
    s = str(text or '').strip()
    if not s:
        return s
    prefix = re.match(r'^\s*[A-Z][A-Za-z.&\' -]{1,22}\b(?:\s+(?:S\.?A\.?S?\.?|B\.?V\.?|LLC|LTD|INC|CO\.?|NV|PLC))?\s+(?=[A-Z])', s)
    if prefix:
        rest = s[prefix.end():].strip()
        if rest and (_PARTY_COMPANY_RE.search(rest) or _PARTY_ADDR_HINT_RE.search(rest)):
            return rest
    return s


_PARTY_LABEL_RE = re.compile(r'(?i)\b(ship\s*to|bill\s*to|sold\s*to|buyer|customer|consignee|deliver\s*to)\b')
_PARTY_STOP_RE = re.compile(
    r'(?i)\b(invoice|facture|subtotal|total|tax|vat|tva|iban|swift|bank|payment|terms|conditions|incoterm|delivery\s+note|purchase\s+order|vendor|seller|amount\s+due)\b'
)
_PARTY_ADDR_HINT_RE = re.compile(
    r'(?i)\b(unit|suite|ste\.?|floor|building|p\.?o\.?\s*box|box|road|rd\.?|street|st\.?|avenue|ave\.?|route|rue|way|boulevard|blvd\.?|drive|dr\.?|lane|ln\.?|lot)\b'
)
_PARTY_COMPANY_RE = re.compile(
    r'(?i)\b(ltd|limited|llc|inc|corp|corporation|company|co\.?|sa\.?|sas\.?|spa\.?|bv\.?|nv\.?|plc|producers?)\b'
)


def _looks_like_party_line(line):
    s = str(line or '').strip()
    if not s or len(s) > 140:
        return False
    if re.search(r'(?i)\b(www\.|https?://|email|fax|phone|tel\.?|ext\.?)\b', s):
        return False
    if _PARTY_STOP_RE.search(s) and not _PARTY_LABEL_RE.search(s):
        return False

    has_company = bool(_PARTY_COMPANY_RE.search(s))
    has_addr = bool(_PARTY_ADDR_HINT_RE.search(s) or re.search(r'\b\d{1,5}\b', s))
    upper_words = re.findall(r'\b[A-Z][A-Z.&-]{2,}\b', s)
    return has_company or has_addr or len(upper_words) >= 2


def _trim_mixed_party_prefix(text):
    s = str(text or '').strip()
    if not s:
        return s
    prefix = re.match(r'^\s*[A-Z][A-Za-z.&\' -]{1,18}\b(?:\s+(?:S\.?A\.?S?\.?|B\.?V\.?|LLC|LTD|INC|CO\.?|NV|PLC))\s+(?=[A-Z])', s)
    if prefix and prefix.end() <= 20:
        rest = s[prefix.end():].strip()
        if rest and (_PARTY_COMPANY_RE.search(rest) or _PARTY_ADDR_HINT_RE.search(rest)):
            return rest
    return s


_PARTY_LABEL_RE = re.compile(r'(?i)\b(ship\s*to|bill\s*to|sold\s*to|buyer|customer|consignee|deliver\s*to)\b')
_PARTY_STOP_RE = re.compile(
    r'(?i)\b(invoice|facture|subtotal|total|tax|vat|tva|iban|swift|bank|payment|terms|conditions|incoterm|delivery\s+note|purchase\s+order|vendor|seller|amount\s+due)\b'
)
_PARTY_ADDR_HINT_RE = re.compile(
    r'(?i)\b(unit|suite|ste\.?|floor|building|p\.?o\.?\s*box|box|road|rd\.?|street|st\.?|avenue|ave\.?|route|rue|way|boulevard|blvd\.?|drive|dr\.?|lane|ln\.?|lot)\b'
)
_PARTY_COMPANY_RE = re.compile(
    r'(?i)\b(ltd|limited|llc|inc|corp|corporation|company|co\.?|sa\.?|sas\.?|spa\.?|bv\.?|nv\.?|plc|producers?)\b'
)


def _looks_like_party_line(line):
    s = str(line or '').strip()
    if not s or len(s) > 140:
        return False
    if re.search(r'(?i)\b(www\.|https?://|email|fax|phone|tel\.?|ext\.?)\b', s):
        return False
    if _PARTY_STOP_RE.search(s) and not _PARTY_LABEL_RE.search(s):
        return False

    has_company = bool(_PARTY_COMPANY_RE.search(s))
    has_addr = bool(_PARTY_ADDR_HINT_RE.search(s) or re.search(r'\b\d{1,5}\b', s))
    upper_words = re.findall(r'\b[A-Z][A-Z.&-]{2,}\b', s)
    return has_company or has_addr or len(upper_words) >= 2


def _trim_mixed_party_prefix(text):
    s = str(text or '').strip()
    if not s:
        return s
    prefix = re.match(r'^\s*[A-Z][A-Za-z.&\' -]{1,18}\b(?:\s+(?:S\.?A\.?S?\.?|B\.?V\.?|LLC|LTD|INC|CO\.?|NV|PLC))\s+(?=[A-Z])', s)
    if prefix and prefix.end() <= 20:
        rest = s[prefix.end():].strip()
        if rest and (_PARTY_COMPANY_RE.search(rest) or _PARTY_ADDR_HINT_RE.search(rest)):
            return rest
    return s


def _clean_party_block(text):
    s = str(text or '').strip()
    if not s:
        return None
    s = s.replace('\n', ' ')
    s = re.sub(r'(?i)^(ship\s*to|bill\s*to|sold\s*to|buyer|customer|consignee|deliver\s*to)\s*[:\-]?\s*', '', s).strip()

    # Remove legal/trade tokens in-place instead of truncating at first occurrence.
    s = re.sub(r'(?i)\b(?:swift|r\.?c\.?s|capital|fob|incoterm|vat|tva|iban|cde\s*n)\b', ' ', s)
    s = re.sub(r'(?i)\b(?:registered office|head office|bank|representant|representative|n.?client)\b', ' ', s)
    s = re.sub(r'(?i)\b\d{1,2}/\d{1,2}/\d{2,4}\b', ' ', s)
    s = re.sub(r'(?i)\bcde\s*n[°o]?\s*\d+\b', ' ', s)
    s = re.sub(r'\s{2,}', ' ', s).strip(' ,;:-')

    if len(s) < 12:
        return None
    if not re.search(r'(?i)(guinep|montego|jamaica|freeport|unit\s*#|bay|way|ltd)', s):
        return None
    return s

def _collect_party_candidate(lines, start_idx, inline_text=''):
    parts = []
    if inline_text:
        inline_text = _trim_mixed_party_prefix(inline_text)
        if inline_text:
            parts.append(inline_text)

    for j in range(start_idx + 1, min(len(lines), start_idx + 5)):
        ln = lines[j].strip()
        if not ln:
            break
        if _PARTY_LABEL_RE.search(ln):
            break
        if not _looks_like_party_line(ln):
            if parts and len(' '.join(parts)) >= 18:
                break
            continue
        parts.append(ln)

    return ' '.join(parts).strip()


def _extract_party_blocks(markdown):
    lines = _markdown_lines(markdown)
    if not lines:
        return {'ship_to': None, 'bill_to': None}

    ship_cands, bill_cands = [], []

    for i, ln in enumerate(lines):
        lo = ln.lower()
        window = ' '.join(lines[i:i+3])

        if any(k in lo for k in ['ship to', 'consignee', 'deliver to']):
            ship_cands.append(window)
            continue
        if any(k in lo for k in ['bill to', 'sold to', 'customer', 'buyer']):
            bill_cands.append(window)
            continue

        if re.search(r'(?i)\bcaribbean\s+producers\b', ln):
            merged = ' '.join(lines[i:i+4])
            ship_cands.append(merged)
            bill_cands.append(merged)

    def _pick(cands):
        best = None
        best_score = -1
        for c in cands:
            cc = _clean_party_block(c)
            if not cc:
                continue
            score = len(cc)
            if re.search(r'(?i)\b1\s*guinep\s*way\b', cc):
                score += 30
            if re.search(r'(?i)\bmontego\b|\bfreeport\b|\bjamaica\b', cc):
                score += 20
            if score > best_score:
                best = cc
                best_score = score
        return best

    return {'ship_to': _pick(ship_cands), 'bill_to': _pick(bill_cands)}

def _safe_float(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    s = normalise_numbers(str(value)).strip()
    if not s:
        return None
    s = s.replace(',', '')
    if not re.search(r'\d', s):
        return None
    try:
        return float(s)
    except Exception:
        return None


def _coerce_numbers(d):
    for f in {"sub_total", "tax", "total", "shipping_handling_charge", "insurance_charge", "discount"}:
        if isinstance(d.get(f), str):
            try:
                d[f] = float(d[f].replace(",", ""))
            except ValueError:
                pass
    return d


_FREIGHT_KW_RE = re.compile(
    r'(?i)\b(ocean\s+freight|air\s+freight|sea\s+freight|freight(?:\s+charge)?|fret|shipping\s+charge|handling\s+charge|surcharge)\b'
)
_INSURANCE_KW_RE = re.compile(r'(?i)\b(insurance|assurance|ins\.?)\b')


def _looks_like_charge_line(item):
    desc = _clean_line_item_description(item.get('description'), item)
    if not desc:
        return False
    if _line_item_pack_hint(desc):
        return False
    if item.get('ean_code') or item.get('material_number'):
        return False

    tokens = re.findall(r'[A-Za-z0-9%./-]+', desc)
    qty = _safe_float(item.get('quantity'))
    la = _safe_float(item.get('line_amount'))
    if la in (None, 0.0):
        return False
    if len(tokens) > 8:
        return False
    if qty not in (None, 0.0, 1.0) and qty > 5:
        return False
    if re.search(r'(?i)\b(?:rose|rouge|white|fries|shrimp|potato|milk|cream|pinot|chardonnay)\b', desc):
        return False
    return True


def _lift_freight_from_line_items(result, items):
    keep = []
    for item in items:
        desc = str(item.get('description') or '')
        la = item.get('line_amount') or 0
        if _FREIGHT_KW_RE.search(desc) and _looks_like_charge_line(item):
            if not result.get('shipping_handling_charge'):
                result['shipping_handling_charge'] = la
                print(f"    Lifted freight to scalar: '{desc[:50]}' = {la}")
            continue
        if _INSURANCE_KW_RE.search(desc) and _looks_like_charge_line(item):
            if not result.get('insurance_charge'):
                result['insurance_charge'] = la
                print(f"    Lifted insurance to scalar: '{desc[:50]}' = {la}")
            continue
        keep.append(item)
    return keep


def _markdown_lines(markdown, aux_text=''):
    lines = []
    for raw in (str(markdown or '') + '\n' + str(aux_text or '')).splitlines():
        line = raw.strip()
        if not line or line.startswith('## Page'):
            continue
        line = line.replace('|', ' ')
        line = re.sub(r'\s{2,}', ' ', line).strip()
        if line:
            lines.append(line)
    return lines


def _desc_key(desc):
    return [t for t in re.findall(r'[a-z0-9]+', str(desc or '').lower())
            if t not in {'the', 'and', 'for', 'with', 'from', 'pack', 'reg', 'po'}]


def _clean_source_description(line, item):
    text = str(line or '')
    if not text:
        return text

    for val in (item.get('ean_code'), item.get('material_number'), item.get('po_number')):
        if val:
            text = re.sub(rf'(?i)\b{re.escape(str(val))}\b', ' ', text)

    text = re.sub(r'^\s*(?:\d+[A-Z]?\s+){1,4}', '', text)
    text = re.sub(r'^\s*(?:BOLT\.?|BOUT\.?|BTL\.?|BLE|AE|FR|ES)\s+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+\d{7,10}\b', ' ', text)

    for val in (item.get('unit_price'), item.get('line_amount')):
        if val is None:
            continue
        try:
            f = float(val)
        except (TypeError, ValueError):
            continue
        candidates = {
            f'{f:.2f}',
            f'{f:.4f}',
            f'{f:.2f}'.replace('.', ','),
            f'{f:.4f}'.replace('.', ','),
        }
        if f >= 1000:
            candidates.add(f'{f:,.2f}')
            candidates.add(f'{f:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.'))
            candidates.add(f'{f:,.2f}'.replace(',', ''))
        for token in sorted(candidates, key=len, reverse=True):
            text = re.sub(rf'(?<!\w){re.escape(token)}(?!\w)', ' ', text)

    text = re.sub(r'\s+(?:AE|FR|ES)\s*$', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+\d+(?:[.,]\d+)?\s*$', '', text)
    text = re.sub(r'^\s*\d+(?:[.,]\d+)?\s+', '', text)
    text = re.sub(r'\s{2,}', ' ', text).strip(' -|,')
    return text.strip()


def _find_best_source_line(item, markdown, aux_text=''):
    lines = _markdown_lines(markdown, aux_text=aux_text)
    if not lines:
        return None
    desc_tokens = set(_desc_key(item.get('description')))
    best_line, best_score = None, -1
    amount = item.get('line_amount')
    pack_re = re.compile(r'(?i)\d+\s*[x/]\s*\d+(?:[.,]\d+)?\s*(?:g|gr|kg|lb|lbs|ml|cl|l)|reg\s*\d+')

    for line in lines:
        lo = line.lower()
        line_tokens = set(_desc_key(line))
        overlap = len(desc_tokens & line_tokens)
        if overlap == 0:
            continue

        score = 0
        if item.get('ean_code') and str(item['ean_code']) in line:
            score += 40
        if item.get('material_number') and str(item.get('material_number')).lower() in lo:
            score += 25

        score += overlap * 4
        score += min(len(line), 120) / 20.0

        if pack_re.search(line):
            score += 10

        if amount is not None:
            try:
                amt = f"{float(amount):.2f}"
                if amt in line or amt.replace('.', ',') in line:
                    score += 8
            except Exception:
                pass

        if score > best_score:
            best_line, best_score = line, score

    return best_line if best_score >= 8 else None



def _enrich_line_items_from_markdown(items, markdown, aux_text=''):
    enriched = []
    for item in items:
        item = dict(item)
        base_desc = str(item.get('description') or '')
        best_line = _find_best_source_line(item, markdown, aux_text=aux_text)
        if best_line:
            candidate = _clean_source_description(best_line, item)
            candidate = _clean_line_item_description(candidate, item)

            base_tokens = set(_desc_key(base_desc))
            cand_tokens = set(_desc_key(candidate))
            overlap = len(base_tokens & cand_tokens)
            added_tokens = len(cand_tokens - base_tokens)
            numeric_tokens = len(re.findall(r'\b\d+(?:[.,]\d+)?\b', candidate))

            id_match = False
            if item.get('ean_code') and re.search(rf"\b{re.escape(str(item.get('ean_code')))}\b", best_line):
                id_match = True
            if item.get('material_number') and re.search(rf"\b{re.escape(str(item.get('material_number')))}\b", best_line, flags=re.IGNORECASE):
                id_match = True

            strong_overlap = overlap >= max(2, min(4, len(base_tokens) or 2))
            informative_gain = added_tokens >= 1 or (_line_item_pack_hint(candidate) and not _line_item_pack_hint(base_desc))
            has_ids = bool(item.get('ean_code') or item.get('material_number'))

            if has_ids:
                candidate_ok = candidate and id_match and strong_overlap and numeric_tokens <= 10 and informative_gain
            else:
                candidate_ok = candidate and strong_overlap and numeric_tokens <= 7 and informative_gain and overlap >= 3

            if candidate_ok and len(candidate) >= max(10, len(base_desc) - 2):
                if len(candidate) > len(base_desc):
                    item['description'] = candidate
        enriched.append(item)
    return enriched

def _enrich_line_items_from_candidates(items, candidate_items):
    base = list(items or [])
    cands = list(candidate_items or [])
    if not base or not cands:
        return base

    out = []
    used = set()
    for item in base:
        item = dict(item)
        best_i, best_s = None, -1.0
        for i, c in enumerate(cands):
            if i in used:
                continue
            sc = _match_line_item_candidates(item, c)
            if sc > best_s:
                best_i, best_s = i, sc

        if best_i is not None and best_s >= 12:
            c = cands[best_i]

            # A) Description replacement only when semantically aligned (or id match).
            cand_desc = _clean_line_item_description(c.get('description'), c)
            cur_desc = _clean_line_item_description(item.get('description'), item)
            base_tokens = set(_desc_key(cur_desc))
            cand_tokens = set(_desc_key(cand_desc))
            overlap = len(base_tokens & cand_tokens)
            id_match = False
            if item.get('ean_code') and c.get('ean_code') and str(item.get('ean_code')) == str(c.get('ean_code')):
                id_match = True
            if item.get('material_number') and c.get('material_number') and str(item.get('material_number')).lower() == str(c.get('material_number')).lower():
                id_match = True
            if cand_desc and len(cand_desc) > len(cur_desc) and (id_match or overlap >= 3):
                item['description'] = cand_desc

            # B) Numeric merge when candidate arithmetic is better for same row amount.
            same_amount = False
            try:
                ia = _safe_float(item.get('line_amount'))
                ca = _safe_float(c.get('line_amount'))
                if ia is not None and ca is not None and abs(ia - ca) <= max(abs(ia) * 0.02, 0.05):
                    same_amount = True
            except Exception:
                same_amount = False
            if same_amount and (not _line_item_arithmetic_ok(item)) and _line_item_arithmetic_ok(c):
                item['quantity'] = c.get('quantity')
                item['unit_price'] = c.get('unit_price')

            used.add(best_i)

        out.append(item)
    return out





def _recover_descriptions_by_identifier(items, markdown='', aux_text=''):
    rows = [dict(it) for it in (items or [])]
    if not rows:
        return rows
    lines = _markdown_lines(markdown, aux_text=aux_text)
    if not lines:
        return rows

    bad_desc_re = re.compile(r'(?i)(invoice declaration|not applicable|terms|conditions|code\s*\(|royal cosun|caribbean producers)')
    used_line_idx = set()

    order = sorted(range(len(rows)), key=lambda i: float(_safe_float(rows[i].get('line_amount')) or 0.0), reverse=True)

    for ridx in order:
        it = rows[ridx]
        ids = []
        if it.get('ean_code'):
            ids.append(str(it.get('ean_code')))
        if it.get('material_number'):
            ids.append(str(it.get('material_number')))
        if not ids:
            continue

        cur_desc = _clean_line_item_description(it.get('description'), it)
        cur_tokens = set(_desc_key(cur_desc))
        best_desc = None
        best_score = -1.0
        best_line_idx = None
        for li, ln in enumerate(lines):
            lo = ln.lower()
            if any(k in lo for k in ['ship to', 'bill to', 'buyer id']):
                continue
            if li in used_line_idx:
                continue
            if not any(i in ln for i in ids):
                continue
            cand = _clean_line_item_description(_clean_source_description(ln, it), it)
            if not cand or len(cand) < 10:
                continue
            if bad_desc_re.search(cand):
                continue

            cand_tokens = set(_desc_key(cand))
            overlap = len(cand_tokens & cur_tokens)
            added = len(cand_tokens - cur_tokens)
            numeric_tokens = len(re.findall(r'\b\d+(?:[.,]\d+)?\b', cand))
            if overlap < max(2, min(4, len(cur_tokens) or 2)):
                continue
            if numeric_tokens > 10:
                continue

            score = len(cand) + (10 if _line_item_pack_hint(cand) else 0) + overlap * 4 + added * 2
            if score > best_score:
                best_score = score
                best_desc = cand
                best_line_idx = li
        if best_desc and len(best_desc) >= len(str(it.get('description') or '')) - 2:
            it['description'] = best_desc
            if best_line_idx is not None:
                used_line_idx.add(best_line_idx)
    return rows

def _enrich_descriptions_with_pack_hints(items, aux_text=''):
    items = list(items or [])
    if not items or not aux_text:
        return items

    lines = _markdown_lines('', aux_text)
    out = []
    pack_re = re.compile(r'(?i)(\b\d+\s*[x/]\s*\d+(?:[.,]\d+)?\s*(?:g|gr|kg|lb|lbs|ml|cl|l)\b|\breg\s*\d+\b)')

    for it in items:
        item = dict(it)
        base = _clean_line_item_description(item.get('description'), item)
        btok = set(_desc_key(base))
        if not btok:
            out.append(item)
            continue

        best = None
        best_score = -1.0
        base_numeric = len(re.findall(r'\b\d+(?:[.,]\d+)?\b', base))
        for ln in lines:
            if not pack_re.search(ln):
                continue
            c = _clean_line_item_description(_clean_source_description(ln, item), item)
            ctok = set(_desc_key(c))
            ov = len(btok & ctok)
            if ov < max(2, int(max(3, len(btok)) * 0.6)):
                continue
            c_numeric = len(re.findall(r'\b\d+(?:[.,]\d+)?\b', c))
            if c_numeric > base_numeric + 4:
                continue
            sc = ov * 10 + len(c) + (8 if _line_item_pack_hint(c) and not _line_item_pack_hint(base) else 0)
            if sc > best_score:
                best_score = sc
                best = c

        if best and len(best) > len(base) + 4 and _line_item_pack_hint(best):
            item['description'] = best
        out.append(item)

    return out

def _normalize_line_item_scales_to_header(items, header):
    rows = [dict(it) for it in (items or [])]
    if not rows:
        return rows

    for it in rows:
        q = _safe_float(it.get('quantity'))
        up = _safe_float(it.get('unit_price'))
        la = _safe_float(it.get('line_amount'))
        pm = _extract_pack_multiplier(it.get('description'))
        if q is not None and q > 0 and q <= 12 and pm and pm >= 6 and up in (None, 0.0) and la in (None, 0.0):
            it['quantity'] = round(q * pm, 4)

    st = _safe_float((header or {}).get('sub_total'))
    tt = _safe_float((header or {}).get('total'))
    target = st
    if tt not in (None, 0.0):
        if target in (None, 0.0):
            target = tt
        else:
            ratio = target / max(abs(tt), 1.0)
            if ratio < 0.6 or ratio > 1.4:
                target = tt
    if target in (None, 0.0):
        return rows

    def _sum_amount(rs):
        vals = [_safe_float(r.get('line_amount')) for r in rs]
        vals = [v for v in vals if v is not None]
        return sum(vals) if vals else 0.0

    curr_sum = _sum_amount(rows)
    if curr_sum <= 0 or curr_sum <= target * 1.35:
        return rows

    improved = True
    while improved:
        improved = False
        base_gap = abs(curr_sum - target)
        best = None
        for i, it in enumerate(rows):
            la = _safe_float(it.get('line_amount'))
            q = _safe_float(it.get('quantity'))
            up = _safe_float(it.get('unit_price'))
            if la in (None, 0.0) or q in (None, 0.0) or up in (None, 0.0):
                continue
            for fac in (10.0, 100.0):
                cand_la = la / fac
                cand_q = q / fac
                if cand_q <= 0:
                    continue
                tol = max(abs(cand_la) * 0.02, 0.05)
                if abs((cand_q * up) - cand_la) > tol:
                    continue
                new_sum = curr_sum - la + cand_la
                new_gap = abs(new_sum - target)
                if new_gap + 1e-9 < base_gap * 0.75:
                    if (best is None) or (new_gap < best[2]):
                        best = (i, fac, new_gap, new_sum)
        if best:
            i, fac, _gap, new_sum = best
            la = _safe_float(rows[i].get('line_amount'))
            q = _safe_float(rows[i].get('quantity'))
            rows[i]['line_amount'] = round(la / fac, 4)
            rows[i]['quantity'] = round(q / fac, 4)
            curr_sum = new_sum
            improved = True

    return rows

def _recover_descriptions_by_amount(items, aux_text=''):
    rows = [dict(it) for it in (items or [])]
    if not rows:
        return rows
    lines = _markdown_lines('', aux_text=aux_text)
    if not lines:
        return rows

    bad_re = re.compile(r'(?i)(invoice declaration|not applicable|terms|conditions|royal cosun|general)')
    num_re = re.compile(r'(?<!\d)(\d{1,3}(?:[.,]\d{2,4})?)(?!\d)')

    for it in rows:
        la = _safe_float(it.get('line_amount'))
        if la is None or la <= 0:
            continue
        best = None
        best_score = -1
        for ln in lines:
            lo = ln.lower()
            if any(k in lo for k in ['ship to', 'bill to', 'buyer id', 'caribbean producers']):
                continue
            nums = []
            for m in num_re.finditer(ln):
                tok = m.group(1).replace(',', '.')
                try:
                    v = float(tok)
                except Exception:
                    continue
                nums.append(v)
            if not nums:
                continue
            if min(abs(v - la) for v in nums) > max(1.0, abs(la) * 0.03):
                continue
            cand = _clean_line_item_description(_clean_source_description(ln, it), it)
            if not cand or len(cand) < 10 or bad_re.search(cand):
                continue
            score = len(cand) + (8 if _line_item_pack_hint(cand) else 0)
            if score > best_score:
                best = cand
                best_score = score
        if best and len(best) >= len(str(it.get('description') or '')) - 2:
            it['description'] = best
    return rows


def _reconcile_header_from_items(header, items, markdown=''):
    result = dict(header)
    amounts = []
    for item in items or []:
        try:
            if item.get('line_amount') is not None:
                amounts.append(float(item['line_amount']))
        except (TypeError, ValueError):
            pass
    if not amounts:
        return result

    items_sum = round(sum(amounts), 2)
    total = _safe_float(result.get('total'))
    sub_total = _safe_float(result.get('sub_total'))
    ship = float(result.get('shipping_handling_charge') or 0.0)
    ins = float(result.get('insurance_charge') or 0.0)
    tax = float(_safe_float(result.get('tax')) or 0.0)

    if sub_total is None:
        result['sub_total'] = items_sum
        sub_total = items_sum
    elif total is not None and abs(float(total) - float(sub_total)) < 0.01 and (ship or ins):
        result['sub_total'] = items_sum
        sub_total = items_sum

    if total in (None, 0, 0.0):
        result['total'] = round(items_sum + ship + ins + tax, 2)
        total = result['total']

    # Strong generic rule: if the invoice total already equals the sum of item rows,
    # freight/insurance are almost certainly hallucinated header fields.
    if total is not None and abs(float(total) - items_sum) <= 0.05:
        result['shipping_handling_charge'] = 0.0
        result['insurance_charge'] = 0.0
        result['sub_total'] = items_sum
        return result

    try:
        st = items_sum if sub_total is None else float(sub_total)
        residual = round(float(total) - st - tax, 2) if total is not None else None
        if residual is None or residual <= 0:
            return result

        tol = max(abs(residual) * 0.005, 0.1)
        current = round(ship + ins, 2)
        if abs(current - residual) <= tol:
            return result

        # Fix common 10x / 0.1x OCR decimal drift on insurance or freight.
        if ship > 0:
            inferred_ins = round(residual - ship, 2)
            if inferred_ins >= 0:
                if abs((ins * 10.0) - inferred_ins) <= max(abs(inferred_ins) * 0.03, 0.2):
                    result['insurance_charge'] = inferred_ins
                    return result
                if abs((ins / 10.0) - inferred_ins) <= max(abs(inferred_ins) * 0.03, 0.2):
                    result['insurance_charge'] = inferred_ins
                    return result

        if ins > 0:
            inferred_ship = round(residual - ins, 2)
            if inferred_ship >= 0:
                if abs((ship * 10.0) - inferred_ship) <= max(abs(inferred_ship) * 0.03, 0.2):
                    result['shipping_handling_charge'] = inferred_ship
                    return result
                if abs((ship / 10.0) - inferred_ship) <= max(abs(inferred_ship) * 0.03, 0.2):
                    result['shipping_handling_charge'] = inferred_ship
                    return result

        # If one field is already plausible, solve the other from arithmetic.
        if ship > 0 and residual >= ship:
            inferred_ins = round(residual - ship, 2)
            if inferred_ins >= 0:
                result['insurance_charge'] = inferred_ins
                return result
        if ins > 0 and residual >= ins:
            inferred_ship = round(residual - ins, 2)
            if inferred_ship >= 0:
                result['shipping_handling_charge'] = inferred_ship
                return result
    except Exception:
        pass

    return result

def _run_role_identification(markdown, letterhead_hint=""):
    doc_label_re = re.compile(
        r"(?i)^(facture|invoice|factura|rechnung|fattura|douane|customs?|proforma|commercial\s+invoice|bordereau|avis|buyer)\b"
    )
    buyer_id_re = re.compile(r"(?i)(buyer\s*[!_]?\s*id|buyer\s*id_|\bbid\b)")
    phone_hint_re = re.compile(r"(?i)\(\d{3}\)\s*[\d\s\-\.]{6,}|\bext\.?\s*\d{3,}\b")

    effective_hint = letterhead_hint
    if letterhead_hint and (
        _is_buyer_text(letterhead_hint)
        or buyer_id_re.search(letterhead_hint)
        or phone_hint_re.search(letterhead_hint)
    ):
        print(f"    Role ID: discarding bad hint '{letterhead_hint[:50]}'")
        effective_hint = ""

    raw = call_slm(ROLE_SYSTEM, build_role_prompt(markdown, effective_hint))
    try:
        result = json.loads(_strip_fences(raw))
        vname = str(result.get("vendor_name") or "")
        if vname and not _is_buyer_text(vname) and not buyer_id_re.search(vname):
            return result
    except Exception:
        pass

    clean = effective_hint.strip()
    if clean and not doc_label_re.match(clean) and not buyer_id_re.search(clean) and not _looks_like_bad_vendor_hint(clean):
        print(f"    Role ID fallback: '{clean[:60]}'")
        return {"vendor_name": clean, "vendor_address": None, "buyer_name": None, "buyer_address": None}
    return {}

def extract_header_json(markdown, page_note="", letterhead_hint=""):
    print(f"    Markdown: {len(markdown):,} chars")
    print("    Pass 1: role ID...")
    roles = _run_role_identification(markdown, letterhead_hint)

    if roles.get("vendor_name"):
        role_note = (
            f"VENDOR (seller): {roles['vendor_name']} | "
            f"{roles.get('vendor_address', 'unknown')}\n"
            f"BUYER (customer): {roles.get('buyer_name', 'unknown')}"
        )
        full_note = f"{role_note}\n\n{page_note}" if page_note else role_note
        print(f"    Vendor: {roles['vendor_name']}")
    else:
        full_note = page_note

    print("    Pass 2: full extraction (header + line items)...")
    raw = call_slm(EXTRACTION_SYSTEM_PROMPT, build_extraction_prompt(markdown, FULL_SCHEMA_STR, full_note))
    try:
        result = json.loads(_strip_fences(raw))
    except json.JSONDecodeError as exc:
        print(f"    JSON parse error: {exc}")
        return {"error": str(exc), "raw": raw}, []

    raw_items = result.pop("line_items", None) or []
    if not isinstance(raw_items, list):
        raw_items = []

    for f in ("vendor_name", "vendor_address", "ship_to", "bill_to"):
        if result.get(f):
            result[f] = str(result[f]).replace("|", "").strip()

    result = _coerce_numbers(result)
    party_blocks = _extract_party_blocks(markdown)

    result["invoice_number"] = _clean_invoice_number(result.get("invoice_number"))
    result["purchase_order_number"] = _clean_po_number(result.get("purchase_order_number"))
    result["invoice_date"] = _normalize_date(result.get("invoice_date"), vendor_address=result.get("vendor_address"))
    result["vendor_phone"] = _validate_phone(result.get("vendor_phone"))
    result["vendor_address"] = _clean_vendor_address(result.get("vendor_address"))

    if not result.get("vendor_name") and roles.get("vendor_name"):
        result["vendor_name"] = roles.get("vendor_name")
    result["vendor_name"] = _repair_vendor_name(result.get("vendor_name"), markdown)

    inferred_vendor_name = _infer_vendor_name_from_markdown(markdown)
    current_vendor_name = str(result.get("vendor_name") or '').strip()
    if inferred_vendor_name and (
        not current_vendor_name
        or len(current_vendor_name.split()) <= 1
        or re.match(r'^(?:[A-Z]\.\s*){4,}', current_vendor_name)
    ):
        result["vendor_name"] = inferred_vendor_name

    if not result.get("vendor_address") and roles.get("vendor_address"):
        result["vendor_address"] = _clean_vendor_address(roles.get("vendor_address"))
    if result.get("vendor_address") and _is_buyer_text(result.get("vendor_address")):
        result["vendor_address"] = None
    if not result.get("vendor_address"):
        inferred_vendor_addr = _infer_vendor_address_from_markdown(markdown, result.get("vendor_name"))
        if inferred_vendor_addr:
            result["vendor_address"] = inferred_vendor_addr

    if not result.get("ship_to"):
        result["ship_to"] = party_blocks.get("ship_to") or roles.get("buyer_address") or roles.get("buyer_name")
    if not result.get("bill_to"):
        result["bill_to"] = party_blocks.get("bill_to") or roles.get("buyer_address") or result.get("ship_to")

    buyer_name = str(roles.get("buyer_name") or '').strip()
    for f in ("ship_to", "bill_to"):
        v = str(result.get(f) or '').strip()
        if not v:
            continue
        if buyer_name and 'caribbean' in buyer_name.lower() and 'caribbean' not in v.lower():
            result[f] = f"{buyer_name} {v}"

    ship_val = str(result.get("ship_to") or "")
    bill_val = str(result.get("bill_to") or "")
    if ship_val and bill_val:
        ship_addr_like = bool(re.search(r'(?i)(guinep|freeport|montego|jamaica|unit\s*#|way)', ship_val))
        bill_addr_like = bool(re.search(r'(?i)(guinep|freeport|montego|jamaica|unit\s*#|way)', bill_val))
        if ship_addr_like and (not bill_addr_like or len(bill_val) + 12 < len(ship_val)):
            result["bill_to"] = ship_val

    for f in ("ship_to", "bill_to"):
        raw = str(result.get(f) or '').strip()
        cleaned = _clean_party_block(raw) if raw else None
        if cleaned:
            result[f] = cleaned
        elif party_blocks.get(f):
            result[f] = party_blocks.get(f)
        elif raw:
            result[f] = re.sub(r'\s{2,}', ' ', raw).strip(' ,;')

    for item in raw_items:
        for f in LINE_ITEM_NUMERIC:
            if isinstance(item.get(f), str):
                try:
                    item[f] = float(item[f].replace(",", ""))
                except ValueError:
                    pass

    raw_items = _lift_freight_from_line_items(result, raw_items)
    validated_items = _validate_line_items(raw_items)
    if validated_items:
        print(f"    LLM extracted {len(validated_items)} line items")

    return result, validated_items

def extract_line_items_only_json(table_text, page_note=""):
    text = str(table_text or '').strip()
    if not text:
        return []
    # Keep the table-only pass small; large OCR dumps cause GPU KV-cache spikes.
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    lines = lines[:80]
    text = '\n'.join(lines)[:5000]
    print("    [5b-TableLLM] Line-item-only extraction...")
    try:
        raw = call_slm(LINE_ITEM_SYSTEM_PROMPT, build_line_item_prompt(text, page_note), max_new_tokens=640)
    except torch.OutOfMemoryError as exc:
        print(f"    [5b-TableLLM] OOM skipped: {exc}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return []
    try:
        parsed = json.loads(_strip_fences(raw))
    except json.JSONDecodeError as exc:
        print(f"    [5b-TableLLM] JSON parse error: {exc}")
        return []

    if isinstance(parsed, dict):
        raw_items = parsed.get('line_items') or []
    elif isinstance(parsed, list):
        raw_items = parsed
    else:
        raw_items = []

    if not isinstance(raw_items, list):
        raw_items = []

    for item in raw_items:
        for f in LINE_ITEM_NUMERIC:
            item[f] = _safe_float(item.get(f))

    validated = _validate_line_items(raw_items)
    if validated:
        print(f"    [5b-TableLLM] Extracted {len(validated)} validated items")
    return validated




SLM helpers ready.


## 21  Full pipeline — `extract_invoice` (v7.2)

**Key fix:** When classified as NATIVE-TEXT but pdfplumber finds zero tables on ALL invoice pages, automatically falls back to TATR OCR. This handles AVIKO which is classified as native-text (Dutch T&C text) but whose invoice table is embedded as a raster image.


In [22]:
def _result_quality_score(header, line_items):
    score = 0.0
    if header.get('vendor_name'):
        score += 1.0
    if header.get('invoice_number'):
        score += 1.5
    t = _safe_float(header.get('total'))
    if t and t > 0:
        score += 1.5
    st = _safe_float(header.get('sub_total'))
    if st and st > 0:
        score += 1.0
    score += min(len(line_items or []), 6) * 1.2
    score += sum(0.6 for it in (line_items or []) if _line_item_arithmetic_ok(it))
    return score


def _result_is_weak(header, line_items):
    h = dict(header or {})
    items = list(line_items or [])
    if len(items) == 0:
        return True

    required_missing = 0
    for key in ('purchase_order_number', 'invoice_date', 'sub_total', 'total'):
        v = h.get(key)
        if v in (None, '', 0, 0.0):
            required_missing += 1
    if required_missing >= 2:
        return True

    total = _safe_float(h.get('total'))
    subtotal = _safe_float(h.get('sub_total'))
    if total in (None, 0.0) or subtotal in (None, 0.0):
        return True

    # If item sum is far from subtotal, hybrid parse is likely wrong.
    vals = [_safe_float(it.get('line_amount')) for it in items]
    vals = [v for v in vals if v is not None]
    if vals:
        s = sum(vals)
        ratio = s / max(abs(subtotal), 1.0)
        if ratio < 0.55 or ratio > 1.6:
            return True

    # If too many scalar nulls, treat as weak.
    scalar_nulls = 0
    for k in ('vendor_address', 'purchase_order_number', 'delivery_note_number', 'vendor_phone'):
        if h.get(k) in (None, ''):
            scalar_nulls += 1
    if scalar_nulls >= 3:
        return True

    return False


def extract_invoice(pdf_path, force_mode=None, _depth=0):
    with tempfile.TemporaryDirectory() as tmp:

        print('  [1/5] Deskew...')
        corrected = preprocess_pdf(pdf_path, tmp)

        print('  [2/5] PDF type probe...')
        pdf_mode, avg_chars, img_fraction = classify_pdf_mode(corrected)
        if force_mode in {'native', 'scanned', 'hybrid'}:
            pdf_mode = force_mode
            print(f"    PDF mode override: {pdf_mode.upper()}")
        scanned = (pdf_mode != 'native')
        if pdf_mode == 'scanned':
            method = 'pytesseract+TATR'
        elif pdf_mode == 'hybrid':
            method = 'hybrid(text+ocr)+TATR'
        else:
            method = 'pdfminer+pdfplumber'

        print(f'  [3/5] Text extraction ({method})...')
        markdown = ''
        docling_used = False
        if pdf_mode in {'scanned', 'hybrid'}:
            md_docling = docling_pages_to_text(corrected)
            q_docling = _invoice_markdown_quality(md_docling)
            if md_docling and q_docling >= 5.0:
                markdown = md_docling
                docling_used = True
                method = f'docling+{method}'
                print(f'    Docling selected (quality={q_docling:.2f}).')
            else:
                if md_docling:
                    print(f'    Docling low quality (quality={q_docling:.2f}) -> fallback to base extractor.')
                else:
                    print('    Docling unavailable/empty -> fallback to base extractor.')

        if not markdown:
            if pdf_mode == 'scanned':
                markdown = scanned_pages_to_text(corrected)
            elif pdf_mode == 'hybrid':
                markdown = hybrid_pages_to_text(corrected)
            else:
                markdown = pdfminer_to_text(corrected)

        print('  [4/5] Page ID + letterhead...')
        filtered_md, page_note, inv_page_indices = identify_invoice_pages(markdown)
        print(f'        {page_note}')
        inv_pages_0 = [i - 1 for i in inv_page_indices]

        anchor_page = inv_pages_0[0] if inv_pages_0 else 0
        letterhead = extract_letterhead_anchor(corrected, scanned, page_num=anchor_page)
        print(f"        Letterhead (page {anchor_page+1}): '{letterhead}'")

        print('  [5/5] Extraction...')
        print('    [5a] Header + line items (LLM)...')
        header, llm_items = extract_header_json(
            filtered_md, page_note=page_note, letterhead_hint=letterhead
        )

        print('    [5b] Structural extraction + candidate selection...')
        raw_items, found_table = pdfplumber_extract_line_items(corrected, inv_pages_0)

        if not scanned and not raw_items:
            print('    [5b-Camelot] native-text invoice ? trying Camelot...')
            raw_items, found_table = camelot_extract_line_items(corrected, inv_pages_0)
            if raw_items:
                method += '+camelot'
            else:
                print('    [5b-Tabula] Camelot found 0 items ? trying Tabula...')
                raw_items, _ = tabula_extract_line_items(corrected, inv_pages_0)
                if raw_items:
                    method += '+tabula'

        if raw_items:
            print(f'        structural extractor found {len(raw_items)} raw items.')
        else:
            if not found_table:
                print('    [5b-TATR] structural text extractors found 0 tables ? trying TATR...')
            else:
                print('    [5b-TATR] structural text extractors found 0 usable items ? trying TATR...')
            raw_items = tatr_extract_line_items(corrected, inv_pages_0)
            if raw_items:
                method += '+TATR-fallback'
            elif scanned:
                print('    [5b-plumber2] TATR found 0 items ? trying pdfplumber on text layer...')
                raw_items, _ = pdfplumber_extract_line_items(corrected, inv_pages_0)
                if raw_items:
                    method += '+plumber-fallback'

        structural_items = _validate_line_items(raw_items)
        if structural_items:
            print(f'        Structural path produced {len(structural_items)} validated items.')

        table_llm_items = []
        cand_table_items = []
        table_text_for_enrich = ''
        if scanned:
            table_text_for_enrich = extract_table_text(corrected, inv_pages_0)

        if scanned and len(structural_items) == 0:
            table_text = table_text_for_enrich
            if table_text:
                cand_table_items = extract_line_items_only_json(table_text, page_note=page_note)
                if _table_items_look_viable(cand_table_items, header):
                    table_llm_items = cand_table_items
                else:
                    print('    [5b-TableLLM] Discarded table items (viability check failed).')

        line_items, item_source, item_score = _choose_line_item_set(
            llm_items, structural_items, header, table_items=table_llm_items
        )
        print(f'        Selected {len(line_items)} line items from {item_source} (score={item_score:.2f}).')
        print(f'        TOTAL: {len(line_items)} line items')

        # Enrich final line item descriptions from the invoice full text.
        # Runs on the chosen set (LLM or structural), so both paths benefit.
        if line_items:
            line_items = _enrich_line_items_from_markdown(line_items, filtered_md, aux_text=table_text_for_enrich)
            line_items = _recover_descriptions_by_identifier(line_items, filtered_md, aux_text=table_text_for_enrich)
            line_items = _recover_descriptions_by_amount(line_items, aux_text=table_text_for_enrich)
            line_items = _enrich_line_items_from_candidates(line_items, cand_table_items)
            line_items = _enrich_descriptions_with_pack_hints(line_items, aux_text=table_text_for_enrich)
            line_items = _validate_line_items(line_items)
            line_items = _normalize_line_item_scales_to_header(line_items, header)
            line_items = _validate_line_items(line_items)

        header = _reconcile_header_from_items(header, line_items, filtered_md)
        final = dict(header)
        final['line_items'] = line_items

        # Hybrid safety fallback: if result is clearly weak, retry with scanned mode and keep better output.
        quality = _result_quality_score(final, line_items)
        if pdf_mode == 'hybrid' and _depth == 0 and (quality < 8.5 or _result_is_weak(header, line_items)):
            print('    [fallback] Hybrid result weak -> retrying scanned mode...')
            alt = extract_invoice(pdf_path, force_mode='scanned', _depth=1)
            alt_q = _result_quality_score(alt.get('header_json') or {}, alt.get('line_items') or [])
            print(f'    [fallback] quality hybrid={quality:.2f}, scanned={alt_q:.2f}')
            if alt_q > quality:
                return alt

    return {
        'json':        final,
        'header_json': header,
        'line_items':  line_items,
        'markdown':    markdown,
        'page_note':   page_note,
        'method':      method,
    }




## 22  Ground-truth evaluator

In [23]:
_SCALAR_FIELDS = [
    "vendor_name", "vendor_address", "invoice_number", "purchase_order_number",
    "invoice_date", "sub_total", "tax", "total", "shipping_handling_charge",
    "insurance_charge", "discount", "ship_to", "bill_to",
    "delivery_note_number", "iban", "vendor_phone",
]
_ITEM_NUMERIC = ["quantity", "unit_price", "line_amount"]
_NUMERIC_TOL  = 0.02   # ±2 cents tolerance for floats


def _norm_str(v) -> str:
    if v is None:
        return ""
    return re.sub(r"\s+", " ", str(v).strip().lower())


def _num_match(a, b, tol=_NUMERIC_TOL) -> bool:
    try:
        return abs(float(a or 0) - float(b or 0)) <= tol
    except (TypeError, ValueError):
        return False


def _item_matches(extracted_item: dict, gt_item: dict) -> tuple[bool, list[str]]:
    """Returns (all_match, list_of_mismatches)."""
    mismatches = []
    # Description: fuzzy — just check key words present
    ext_desc = _norm_str(extracted_item.get("description"))
    gt_desc  = _norm_str(gt_item.get("description"))
    if ext_desc != gt_desc:
        mismatches.append(f"description: got '{ext_desc[:60]}' expected '{gt_desc[:60]}'")
    for f in _ITEM_NUMERIC:
        if not _num_match(extracted_item.get(f), gt_item.get(f)):
            mismatches.append(f"{f}: got {extracted_item.get(f)} expected {gt_item.get(f)}")
    return len(mismatches) == 0, mismatches


def evaluate_invoice(extracted: dict, gt: dict, label: str = "") -> dict:
    """
    Compare one extracted invoice dict against one ground-truth dict.
    Returns a results dict and prints a detailed breakdown.
    """
    PAD = 30
    field_results = {}

    print(f"\n{'─'*70}")
    print(f"  {label}")
    print(f"{'─'*70}")

    # ── Scalar fields ────────────────────────────────────────────────────────
    scalar_pass = 0
    for field in _SCALAR_FIELDS:
        ext_val = extracted.get(field)
        gt_val  = gt.get(field)

        # Numeric comparison
        if isinstance(gt_val, (int, float)) or isinstance(ext_val, (int, float)):
            ok = _num_match(ext_val, gt_val)
        else:
            ok = _norm_str(ext_val) == _norm_str(gt_val)

        status = "✅" if ok else "❌"
        field_results[field] = ok
        if ok:
            scalar_pass += 1
            print(f"  {status} {field:<{PAD}} {str(ext_val)[:60]}")
        else:
            print(f"  {status} {field:<{PAD}} got='{str(ext_val)[:40]}' | expected='{str(gt_val)[:40]}'")

    # ── Line items ───────────────────────────────────────────────────────────
    ext_items = extracted.get("line_items", []) or []
    gt_items  = gt.get("line_items", []) or []

    print(f"\n  line_items: extracted {len(ext_items)}, expected {len(gt_items)}")
    item_pass = 0
    matched_gt = set()

    for ei, ext_item in enumerate(ext_items):
        # Find the best matching GT item (by description)
        best_gt_idx, best_overlap = None, -1
        for gi, gt_item in enumerate(gt_items):
            if gi in matched_gt:
                continue
            ext_words = set(_norm_str(ext_item.get("description")).split())
            gt_words  = set(_norm_str(gt_item.get("description")).split())
            overlap   = len(ext_words & gt_words)
            if overlap > best_overlap:
                best_overlap, best_gt_idx = overlap, gi

        if best_gt_idx is None:
            print(f"    ❌ Item {ei+1}: no matching GT item")
            continue

        matched_gt.add(best_gt_idx)
        ok, mismatches = _item_matches(ext_item, gt_items[best_gt_idx])
        if ok:
            item_pass += 1
            print(f"    ✅ Item {ei+1}: {_norm_str(ext_item.get('description'))[:55]}")
        else:
            print(f"    ❌ Item {ei+1}: {_norm_str(ext_item.get('description'))[:55]}")
            for m in mismatches:
                print(f"         → {m}")

    unmatched_gt = [i for i in range(len(gt_items)) if i not in matched_gt]
    for gi in unmatched_gt:
        print(f"    ❌ Missing GT item {gi+1}: {_norm_str(gt_items[gi].get('description'))[:55]}")

    item_total = max(len(gt_items), len(ext_items), 1)
    items_score = item_pass / item_total if gt_items else 1.0

    scalar_total = len(_SCALAR_FIELDS)
    print(f"\n  Scalar fields : {scalar_pass}/{scalar_total}")
    print(f"  Line items    : {item_pass}/{len(gt_items) or len(ext_items) or 1}")
    overall = (scalar_pass / scalar_total + items_score) / 2
    print(f"  Overall score : {overall*100:.1f}%")

    return {
        "scalar_pass":  scalar_pass,
        "scalar_total": scalar_total,
        "item_pass":    item_pass,
        "item_total":   len(gt_items),
        "overall":      overall,
        "field_results": field_results,
    }


def print_eval_summary(eval_results: list[dict], labels: list[str]):
    print(f"\n{'='*70}")
    print("  EVALUATION SUMMARY")
    print(f"{'='*70}")
    print(f"  {'INVOICE':<45} {'SCALAR':>8} {'ITEMS':>8} {'SCORE':>8}")
    print(f"  {'-'*45} {'-'*8} {'-'*8} {'-'*8}")
    for label, r in zip(labels, eval_results):
        s = f"{r['scalar_pass']}/{r['scalar_total']}"
        i = f"{r['item_pass']}/{r['item_total']}"
        o = f"{r['overall']*100:.1f}%"
        print(f"  {label:<45} {s:>8} {i:>8} {o:>8}")
    avg = sum(r["overall"] for r in eval_results) / len(eval_results)
    print(f"\n  Average overall score: {avg*100:.1f}%")
    print(f"{'='*70}")

print("Evaluator defined.")

Evaluator defined.


## 23  Single-invoice demo (AVIKO)

In [24]:
print("=" * 70)
print("  AVIKO -- single invoice demo (v7 split architecture)")
print("=" * 70)

result = extract_invoice(INVOICE_PATHS[0])

print("\n-- Header JSON --")
print(json.dumps(result["header_json"], indent=2, ensure_ascii=False))
print("\n-- Line items --")
print(json.dumps(result["line_items"], indent=2, ensure_ascii=False))
print("\n-- Method:", result["method"])

evaluate_invoice(result["json"], GROUND_TRUTH[0], label="AVIKO")

  AVIKO -- single invoice demo (v7 split architecture)
  [1/5] Deskew...
    Pre-process: no corrections needed
  [2/5] PDF type probe...
    PDF type: SCANNED (image=90% > 80% and avg 2593 chars/page < 4000 — CLASS C: raster with text overlay)
  [3/5] Text extraction (pytesseract+TATR)...
  [4/5] Page ID + letterhead...
        Document has 4 pages. Vendor invoice pages (USE THESE): [2, 3]. Excluded pages (purchase orders / logistics docs): [4]. Extract ALL fields ONLY from the vendor invoice pages.


Passing `generation_config` together with generation-related arguments=({'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


        Letterhead (page 2): 'Alle andere algemene voorwaarden worden uitdrukkelijk van de hand gewezen.'
  [5/5] Extraction...
    [5a] Header + line items (LLM)...
    Markdown: 8,696 chars
    Pass 1: role ID...


Both `max_new_tokens` (=1536) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=1536) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Vendor: Aviko B.V.
    Pass 2: full extraction (header + line items)...
    Fix(B 1380.0x): 'CAR Aviko Str. cut fries 3/8' qty=1.0->1380.00
    LLM extracted 3 line items
    [5b] Structural extraction + candidate selection...
    pdfplumber: no scoreable table on page 2
    pdfplumber: no scoreable table on page 3
    pdfplumber: 0 line items (found_any=False)
    [5b-TATR] structural text extractors found 0 tables ? trying TATR...
    TATR p2: 79 rows x 6 cols — building grid via word assignment
    Grid has 57 non-empty rows out of 79
      row   0: ['Op alle', 'overeenkomsten met', 'en/of (rechts)handelingen van', 'Aviko m.b.t. verkoop van goederen, diensten', 'en/of werkzaamheden', 'zijn de algemene verkaopvoorwaarden en']
      row   2: ['arbitragereglernent', 'van de', 'V.A.VI. (de vereniging voor de', 'Aardappelverwerkende industrie) — gedeponeerd', 'onder dossiernummer', '7/2016 bij de Kamer van Koophandel - van']
      row   4: ['toepassing.', 'Deze algemene', 'verkoopvoo

Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=640) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Table-text p3: selected OCR crop (score=92)
    [5b-TableLLM] Line-item-only extraction...
    [5b-TableLLM] Discarded table items (viability check failed).
        Candidate llm        score=25.50 :: count=3 arithmetic=3/3 desc=4.7 ids=1.00 subtotal_bonus=4.34 subtotal_penalty=0.00 zero_penalty=0.00
        Candidate llm+struct score=25.50 :: count=3 arithmetic=3/3 desc=4.7 ids=1.00 subtotal_bonus=4.34 subtotal_penalty=0.00 zero_penalty=0.00
        Candidate struct     score=-1000000.00 :: empty
        Candidate struct+llm score=25.50 :: count=3 arithmetic=3/3 desc=4.7 ids=1.00 subtotal_bonus=4.34 subtotal_penalty=0.00 zero_penalty=0.00
        Selected 3 line items from llm (score=25.50).
        TOTAL: 3 line items

-- Header JSON --
{
  "vendor_name": "Aviko B.V.",
  "vendor_address": "Dr. A. Ariénsstraat 28 7221 CD Steenderen / P.O, Box 8 7220 AA Steenderen Holland",
  "invoice_number": "93184726",
  "purchase_order_number": "POH112330",
  "invoice_date": "2024-03-06",
  "su

{'scalar_pass': 12,
 'scalar_total': 16,
 'item_pass': 0,
 'item_total': 3,
 'overall': 0.375,
 'field_results': {'vendor_name': True,
  'vendor_address': False,
  'invoice_number': True,
  'purchase_order_number': True,
  'invoice_date': True,
  'sub_total': True,
  'tax': True,
  'total': True,
  'shipping_handling_charge': True,
  'insurance_charge': True,
  'discount': True,
  'ship_to': False,
  'bill_to': False,
  'delivery_note_number': True,
  'iban': False,
  'vendor_phone': True}}

## 24  Inspect intermediate Markdown

In [25]:
print("-- Raw Markdown (first 3000 chars) --")
print(result["markdown"][:3000])
print("\n-- Page note --")
print(result["page_note"])

-- Raw Markdown (first 3000 chars) --
## Page 1
— = r
7 ‘| SIFKO
IXY F9°S&
Commercial invoice Page: 1 of 1
Information ees Sse ‘Billte panty ne. G4TF eee
Invoice no. 93184726
Invoice date 06.03.2024 Caribbean Producers Jamaica Ltd
Order no. supplier 2974983 One Guinep Way
Delivery note no. 83661626 Montego Freeport
- MONTEGO BAY
Purchase order no, POH112330 JAMAICA
Purchase order date 19.02.2024
Incoterms CIF, Kingston-Port
Currency EUR
Terms of payment Net Bank (45 days)
Payment due date 20.04.2024 gaye ergata recat ccccsnennccnass eas cc nnat anaes anne eccecaamaatanatstosmonneaaaeamesan soseesosee
‘SHO te party no, GBR ee scenes
Caribbean Producers Jamaica Ltd
Container-id. TTNU8071689 One Guinep Way
Seal id. 1064780 Montego Freeport
Shipping details Caucedo express o.s. ic BAY
Rotterdam-port ETD 12.03.2024
Kingston-port ETA 01.04.2024
Goods issue date 06.03.2024
Req, delivery date 06.03.2024
EANno./ = —s Material iy Material description Stat. Code =———s~Price ‘Digcounty ‘Amount VAT

## 25  Batch extraction + ground-truth evaluation

In [26]:
batch_results, eval_results, labels = {}, [], []

for idx, pdf_path in enumerate(INVOICE_PATHS):
    key   = os.path.basename(pdf_path)
    gt    = GROUND_TRUTH[idx]
    label = gt["vendor_name"]

    print(f"\n{'='*70}")
    print(f"  [{idx+1}/9]  {key}")
    print(f"{'='*70}")

    try:
        res = extract_invoice(pdf_path)
    except Exception as exc:
        import traceback
        print(f"  PIPELINE ERROR: {exc}")
        traceback.print_exc()
        batch_results[key] = {"error": str(exc)}
        eval_results.append({"scalar_pass": 0, "scalar_total": len(_SCALAR_FIELDS),
                              "item_pass": 0, "item_total": len(gt.get("line_items", [])),
                              "overall": 0.0, "field_results": {}})
        labels.append(label)
        continue

    batch_results[key] = res
    print("\n-- Extracted JSON --")
    print(json.dumps(res["json"], indent=2, ensure_ascii=False))
    print(f"-- Method: {res['method']}")
    eval_results.append(evaluate_invoice(res["json"], gt, label=label))
    labels.append(label)

print_eval_summary(eval_results, labels)


  [1/9]  Direct - AVIKO - G24000153.pdf
  [1/5] Deskew...
    Pre-process: no corrections needed
  [2/5] PDF type probe...
    PDF type: SCANNED (image=90% > 80% and avg 2593 chars/page < 4000 — CLASS C: raster with text overlay)
  [3/5] Text extraction (pytesseract+TATR)...
  [4/5] Page ID + letterhead...
        Document has 4 pages. Vendor invoice pages (USE THESE): [2, 3]. Excluded pages (purchase orders / logistics docs): [4]. Extract ALL fields ONLY from the vendor invoice pages.


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=1536) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


        Letterhead (page 2): 'Alle andere algemene voorwaarden worden uitdrukkelijk van de hand gewezen.'
  [5/5] Extraction...
    [5a] Header + line items (LLM)...
    Markdown: 8,696 chars
    Pass 1: role ID...


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=1536) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Vendor: Aviko B.V.
    Pass 2: full extraction (header + line items)...
    Fix(B 1380.0x): 'CAR Aviko Str. cut fries 3/8' qty=1.0->1380.00
    LLM extracted 3 line items
    [5b] Structural extraction + candidate selection...
    pdfplumber: no scoreable table on page 2
    pdfplumber: no scoreable table on page 3
    pdfplumber: 0 line items (found_any=False)
    [5b-TATR] structural text extractors found 0 tables ? trying TATR...
    TATR p2: 79 rows x 6 cols — building grid via word assignment
    Grid has 57 non-empty rows out of 79
      row   0: ['Op alle', 'overeenkomsten met', 'en/of (rechts)handelingen van', 'Aviko m.b.t. verkoop van goederen, diensten', 'en/of werkzaamheden', 'zijn de algemene verkaopvoorwaarden en']
      row   2: ['arbitragereglernent', 'van de', 'V.A.VI. (de vereniging voor de', 'Aardappelverwerkende industrie) — gedeponeerd', 'onder dossiernummer', '7/2016 bij de Kamer van Koophandel - van']
      row   4: ['toepassing.', 'Deze algemene', 'verkoopvoo

...

        Letterhead (page 2): 'po DAY OS Nome'
  [5/5] Extraction...
    [5a] Header + line items (LLM)...
    Markdown: 2,985 chars
    Pass 1: role ID...


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=1536) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Vendor: MINUTY SAS
    Pass 2: full extraction (header + line items)...
    invoice_number: rejected BL 'BL00527870'
    Fix(B 0.1x): 'CHATEAU MINUTY OR - A.O.P COTES DE PROVENCE ROSE ET OR ROSE ' qty=1980.0->180.00
    Fix(B 12.0x): 'M DE MINUTY AOP COTES DE PROVENCE ROSE 0,75 L, ALCOHOL 13% 2' qty=150.0->1800.00
    LLM extracted 2 line items
    [5b] Structural extraction + candidate selection...
    pdfplumber: no scoreable table on page 2
    pdfplumber: 0 line items (found_any=False)
    [5b-TATR] structural text extractors found 0 tables ? trying TATR...
    TATR p2: 45 rows x 8 cols — building grid via word assignment
    Grid has 23 non-empty rows out of 45
      row   0: ['ee |', '', '', '', '', '', '', '']
      row   1: ['CARIBBEAN PRODUCERS JAMAICA LT 1 GUINEP WAY', '', '', '', '', '', '', '']
      row   2: ['MONTEGO FREEPORT [Date', '', '', '', '', '', '', '']
      row   3: ['MONTEGO BAY ST. JAMES', '', '', '', '', '', '', '']
      row   4: ['ksewell', '', '', '', 

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=640) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Table-text p2: selected OCR crop (score=92)
    [5b-TableLLM] Line-item-only extraction...
    [5b-TableLLM] Extracted 4 validated items
    [5b-TableLLM] Discarded table items (viability check failed).
        Candidate llm        score=24.00 :: count=2 arithmetic=2/2 desc=15.0 ids=0.00 subtotal_bonus=0.00 subtotal_penalty=0.00 zero_penalty=0.00
        Candidate llm+struct score=24.00 :: count=2 arithmetic=2/2 desc=15.0 ids=0.00 subtotal_bonus=0.00 subtotal_penalty=0.00 zero_penalty=0.00
        Candidate struct     score=-1000000.00 :: empty
        Candidate struct+llm score=24.00 :: count=2 arithmetic=2/2 desc=15.0 ids=0.00 subtotal_bonus=0.00 subtotal_penalty=0.00 zero_penalty=0.00
        Selected 2 line items from llm (score=24.00).
        TOTAL: 2 line items

-- Extracted JSON --
{
  "vendor_name": "MINUTY SAS",
  "vendor_address": "ZAC Niet. i da LY, LY 0, 2491 route de la Berle - 83580 Gassin - France",
  "invoice_number": null,
  "purchase_order_number": "PO H110049",


## 26  Field-level heatmap

In [27]:
field_matrix = {f: [] for f in _SCALAR_FIELDS}
for ev in eval_results:
    for f in _SCALAR_FIELDS:
        field_matrix[f].append(ev["field_results"].get(f, False))

print(f"\n  {'FIELD':<30} " + "  ".join(f"{labels[i][:6]:>7}" for i in range(len(labels))))
print(f"  {'-'*30} " + "  ".join(["-"*7] * len(labels)))
for field, results in field_matrix.items():
    row = "  ".join("   OK  " if r else "   XX  " for r in results)
    pass_count = sum(results)
    print(f"  {field:<30} {row}   ({pass_count}/{len(results)})")



  FIELD                           Aviko    Nemco    Quirch   BARON    BARON    de Lad   LES VI   HOBNOB   MINUTY
  ------------------------------ -------  -------  -------  -------  -------  -------  -------  -------  -------
  vendor_name                       OK       OK       OK       XX       XX       XX       XX       XX       XX     (3/9)
  vendor_address                    XX       XX       XX       XX       XX       XX       XX       XX       XX     (0/9)
  invoice_number                    OK       OK       OK       OK       OK       OK       OK       XX       XX     (7/9)
  purchase_order_number             OK       XX       OK       OK       OK       OK       XX       XX       OK     (6/9)
  invoice_date                      OK       OK       OK       OK       OK       OK       OK       XX       XX     (7/9)
  sub_total                         OK       OK       OK       OK       OK       OK       OK       XX       OK     (8/9)
  tax                               OK       OK

## 27  GPU memory check

In [28]:
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc  = torch.cuda.memory_allocated(i) / 1024**3
        reserv = torch.cuda.memory_reserved(i)  / 1024**3
        total  = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"GPU {i}: {alloc:.1f} GB alloc / {reserv:.1f} GB reserved / {total:.0f} GB total")
else:
    print("No CUDA GPUs detected.")


GPU 0: 1.6 GB alloc / 2.0 GB reserved / 15 GB total
GPU 1: 3.8 GB alloc / 6.0 GB reserved / 15 GB total


## 28  Tuning guide (v7.2)

### Key diagnostics to check in each run

**PDF classification:** Look for `'PDF type: SCANNED (image=90% > 80% and avg 2593 chars/page < 4000 — CLASS C)'`.
If AVIKO still goes to pdfminer, lower the `4000` threshold or raise the `0.80` image fraction.

**Letterhead:** `Letterhead (page N): 'Aviko B.V'` — correct. If you see `'tee |'` or `'CARIBBEAN PRODUCERS'` → the OCR on the invoice page is still reading a different block first. Raise the top-strip fraction from 0.40 → 0.55 in `_top_lines_tesseract`, or add the offending string to `_BUYER_STRINGS` / `_BAD_ANCHOR_RE`.

**TATR column mapping:** `TATR best header row=N mapping={...}`. If the mapping has fewer than 2 key fields, `_mapping_is_usable` triggers positional fallback. Check `'Positional col mapping: {3: line_amount, 2: unit_price, 1: quantity, 4: description}'` to verify the positional assignment is correct.

**OCR fallback:** `'[5b-fallback] pdfplumber found 0 tables — falling back to TATR OCR'` means the PDF was classified as native-text but the invoice table is in an embedded image.

### Known remaining issues

- **AVIKO invoice_number** (`93184726` vs `99194726`): Even with OCR, digit transpositions are possible. The number is in a scanned image; tesseract OCR at 300 DPI occasionally misreads `3→9` and `8→9`. Raising `RASTER_DPI` to 400 may help.

- **vendor_address mismatch**: AVIKO has two addresses — physical (`Dr. A. Ariënsstraat`) and PO Box (`P.O. Box 8`). The GT uses the PO Box. The LLM reads whichever appears first in the OCR output. This is a GT quality ambiguity, not a pipeline error.

- **QUIRCH/HOBNOB invoice_date DD/MM vs MM/DD**: `_normalize_date` uses `dayfirst=True` (European convention). QUIRCH is a US company so its dates are MM/DD. Detect US-based vendors (address contains US state abbreviation) and pass `dayfirst=False`.
